In [11]:
# MTFTT (Proposed Model) with binary


import os, json, time, math, random
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    matthews_corrcoef, cohen_kappa_score
)
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm

from imblearn.over_sampling import SMOTE


# PLOTTING CONFIG
@dataclass
class PLOT_CFG:
    font_family: str = "DejaVu Sans"
    base_fontsize: int = 13
    title_size: int = 9
    label_size: int = 15
    tick_size: int = 13
    legend_size: int = 12

    bold_all_text: bool = True
    title_weight: str = "bold"
    label_weight: str = "bold"
    tick_weight: str = "bold"
    legend_weight: str = "bold"

    dpi_png: int = 350
    save_bbox: str = "tight"
    fig_w: float = 8.5
    fig_h: float = 4.2

    grid_on: bool = True
    grid_alpha: float = 0.25
    spine_linewidth: float = 1.3
    tick_length: float = 5.0
    tick_width: float = 1.2

    line_width: float = 2.2
    vline_width: float = 1.4
    marker_size: float = 5.0

    legend_frame: bool = False
    legend_loc: str = "best"
    legend_ncols: int = 1

    rotate_xticks_deg: int = 25
    cmap_confusion: str = "Blues"


PLOT = PLOT_CFG()


def apply_plot_style(cfg: PLOT_CFG = PLOT) -> None:
    plt.style.use("ggplot")
    plt.rcParams.update({
        "font.family": cfg.font_family,
        "font.size": cfg.base_fontsize,
        "axes.titlesize": cfg.title_size,
        "axes.titleweight": cfg.title_weight if cfg.bold_all_text else "normal",
        "axes.labelsize": cfg.label_size,
        "axes.labelweight": cfg.label_weight if cfg.bold_all_text else "normal",
        "xtick.labelsize": cfg.tick_size,
        "ytick.labelsize": cfg.tick_size,
        "legend.fontsize": cfg.legend_size,
        "figure.dpi": 120,
        "savefig.bbox": cfg.save_bbox,
        "lines.linewidth": cfg.line_width,
        "lines.markersize": cfg.marker_size,
    })


def _make_fig(size: Optional[tuple] = None):
    if size is None:
        size = (PLOT.fig_w, PLOT.fig_h)
    fig, ax = plt.subplots(figsize=size)
    return fig, ax


def style_axes(ax):
    if PLOT.grid_on:
        ax.grid(True, alpha=PLOT.grid_alpha)
    for spine in ax.spines.values():
        spine.set_linewidth(PLOT.spine_linewidth)
    ax.tick_params(axis="both", which="major",
                   labelsize=PLOT.tick_size,
                   length=PLOT.tick_length,
                   width=PLOT.tick_width)
    if PLOT.bold_all_text:
        for lab in ax.get_xticklabels():
            lab.set_fontweight(PLOT.tick_weight)
        for lab in ax.get_yticklabels():
            lab.set_fontweight(PLOT.tick_weight)


def _bold_legend(ax):
    leg = ax.legend(loc=PLOT.legend_loc, frameon=PLOT.legend_frame, ncols=PLOT.legend_ncols)
    if leg is not None and PLOT.bold_all_text:
        for t in leg.get_texts():
            t.set_fontweight(PLOT.legend_weight)


def _bold_title_labels(ax, title: str, xlabel: str, ylabel: str):
    ax.set_title(title, fontsize=PLOT.title_size,
                 fontweight=PLOT.title_weight if PLOT.bold_all_text else "normal")
    ax.set_xlabel(xlabel, fontsize=PLOT.label_size,
                  fontweight=PLOT.label_weight if PLOT.bold_all_text else "normal")
    ax.set_ylabel(ylabel, fontsize=PLOT.label_size,
                  fontweight=PLOT.label_weight if PLOT.bold_all_text else "normal")


def save_png(fig, plots_dir: Path, name: str):
    fig.savefig(plots_dir / f"{name}.png", dpi=PLOT.dpi_png, bbox_inches=PLOT.save_bbox)


def plot_lines(dfh: pd.DataFrame, x: str, ys: List[str], labels: List[str],
               title: str, xlabel: str, ylabel: str,
               stage_boundaries: Optional[List[int]] = None):

    fig, ax = _make_fig()
    for y, lab in zip(ys, labels):
        ax.plot(dfh[x], dfh[y], label=lab, linewidth=PLOT.line_width)
    if stage_boundaries:
        for b in stage_boundaries:
            ax.axvline(b, linestyle="--", linewidth=PLOT.vline_width, alpha=0.65)
    _bold_title_labels(ax, title, xlabel, ylabel)
    _bold_legend(ax)
    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_confusion(cm: np.ndarray, class_names: List[str], normalize: bool, title: str):
    mat = cm.astype(np.float32)
    if normalize:
        mat = mat / (mat.sum(axis=1, keepdims=True) + 1e-12)

    fig, ax = _make_fig(size=(7.2, 6.2))
    im = ax.imshow(mat, interpolation="nearest", cmap=PLOT.cmap_confusion)
    ax.set_title(title, fontsize=PLOT.title_size,
                 fontweight=PLOT.title_weight if PLOT.bold_all_text else "normal")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if PLOT.bold_all_text:
        cbar.ax.tick_params(labelsize=PLOT.tick_size)
        for lab in cbar.ax.get_yticklabels():
            lab.set_fontweight(PLOT.tick_weight)

    tick = np.arange(len(class_names))
    ax.set_xticks(tick); ax.set_yticks(tick)
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=PLOT.tick_size)
    ax.set_yticklabels(class_names, fontsize=PLOT.tick_size)

    ax.set_xlabel("Predicted", fontsize=PLOT.label_size,
                  fontweight=PLOT.label_weight if PLOT.bold_all_text else "normal")
    ax.set_ylabel("True", fontsize=PLOT.label_size,
                  fontweight=PLOT.label_weight if PLOT.bold_all_text else "normal")

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            txt = f"{mat[i, j]:.2f}" if normalize else str(int(mat[i, j]))
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=PLOT.tick_size,
                    fontweight="bold" if PLOT.bold_all_text else "normal")

    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_barh_top(values: np.ndarray, names: List[str], title: str, topk: int = 15):
    idx = np.argsort(-values)[:topk][::-1]
    fig, ax = _make_fig(size=(8.8, 5.4))
    ax.barh([names[i] for i in idx], [values[i] for i in idx])
    _bold_title_labels(ax, title, "Importance", "")
    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_reliability(bin_conf: np.ndarray, bin_acc: np.ndarray, title: str):
    fig, ax = _make_fig(size=(6.6, 5.4))
    ax.plot(bin_conf, bin_acc, marker="o", linewidth=PLOT.line_width, label="Empirical")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=PLOT.vline_width, label="Ideal")
    _bold_title_labels(ax, title, "Confidence", "Accuracy")
    _bold_legend(ax)
    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_per_class_f1(rep: Dict, class_names: List[str], title: str):
    f1s = [rep[c]["f1-score"] for c in class_names]
    fig, ax = _make_fig(size=(8.8, 4.4))
    ax.bar(class_names, f1s)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=PLOT.rotate_xticks_deg)
    _bold_title_labels(ax, title, "", "F1-score")
    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_decision_curve(dca_df: pd.DataFrame, title: str):
    fig, ax = _make_fig(size=(7.8, 5.2))
    ax.plot(dca_df["threshold"], dca_df["net_benefit_model"],
            linewidth=PLOT.line_width, label="MTFTT (model)")
    ax.plot(dca_df["threshold"], dca_df["net_benefit_treat_all"],
            linewidth=PLOT.line_width, linestyle="--", label="Treat-all")
    ax.plot(dca_df["threshold"], dca_df["net_benefit_treat_none"],
            linewidth=PLOT.line_width, linestyle=":", label="Treat-none")
    _bold_title_labels(ax, title, "Threshold probability", "Net benefit")
    _bold_legend(ax)
    style_axes(ax)
    fig.tight_layout()
    return fig


# XAI PLOTS
def plot_integrated_gradients_sample(attributions, feature_names, sample_id, true_class, pred_class, class_names):
    attr_vals = np.asarray(attributions).squeeze()
    idx = np.argsort(np.abs(attr_vals))[::-1][:15]
    fig, ax = _make_fig(size=(9.5, 6.0))
    colors = ['#d62728' if a < 0 else '#2ca02c' for a in attr_vals[idx]]
    ax.barh(range(len(idx)), attr_vals[idx], color=colors)
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feature_names[i] for i in idx], fontsize=PLOT.tick_size)
    title = (f"Integrated Gradients (Sample {sample_id})\n"
             f"True: {class_names[true_class]} | Pred: {class_names[pred_class]}")
    _bold_title_labels(ax, title, "Attribution Score", "")
    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_integrated_gradients_global(all_attributions, feature_names, title):
    all_attributions = np.asarray(all_attributions)
    mean_abs_attr = np.mean(np.abs(all_attributions), axis=0)
    idx = np.argsort(mean_abs_attr)[::-1][:15]
    fig, ax = _make_fig(size=(9.0, 5.5))
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(idx)))
    ax.barh(range(len(idx)), mean_abs_attr[idx], color=colors)
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feature_names[i] for i in idx], fontsize=PLOT.tick_size)
    _bold_title_labels(ax, title, "Mean |Attribution|", "")
    style_axes(ax)
    fig.tight_layout()
    return fig


def plot_multihead_attention_grid(attn_weights, feature_names, layer_idx, n_heads=8):
    n_heads_actual = attn_weights.shape[0]
    n_cols = 4
    n_rows = int(np.ceil(n_heads_actual / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else np.array([axes]).flatten()

    cmap = LinearSegmentedColormap.from_list('attn', ['white', '#3498db', '#e74c3c'])
    for head_idx in range(n_heads_actual):
        ax = axes[head_idx]
        attn_head = attn_weights[head_idx]
        vmax = max(1e-9, float(np.max(attn_head)))
        im = ax.imshow(attn_head, cmap=cmap, aspect='auto', vmin=0, vmax=vmax)
        tick_indices = list(range(0, len(feature_names), max(1, len(feature_names) // 10)))
        tick_labels = [feature_names[i][:8] for i in tick_indices]
        ax.set_xticks(tick_indices); ax.set_yticks(tick_indices)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(tick_labels, fontsize=7)
        ax.set_title(f"Head {head_idx + 1}", fontsize=10, fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for idx in range(n_heads_actual, len(axes)):
        axes[idx].axis('off')

    fig.suptitle(f"Multi-Head Attention Grid (Layer {layer_idx + 1})",
                 fontsize=14, fontweight='bold', y=0.995)
    fig.tight_layout()
    return fig


def plot_attention_heatmap_averaged(attn_weights, feature_names, title="Layer Attention (Head-Averaged)"):
    attn_avg = attn_weights.mean(axis=0)  # [F,F]
    fig, ax = _make_fig(size=(11, 10))
    im = ax.imshow(attn_avg, cmap='YlOrRd', aspect='auto')
    n_show = min(len(feature_names), 20)
    step = max(1, len(feature_names) // n_show)
    tick_indices = list(range(0, len(feature_names), step))
    tick_labels = [feature_names[i][:10] for i in tick_indices]
    ax.set_xticks(tick_indices); ax.set_yticks(tick_indices)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(tick_labels, fontsize=9)
    ax.set_xlabel("To (Key)", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_ylabel("From (Query)", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_title(title, fontsize=PLOT.title_size, fontweight=PLOT.title_weight)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Attention Weight", fontsize=PLOT.label_size)
    fig.tight_layout()
    return fig


def plot_attention_rollout_matrix(rollout_mat: np.ndarray, feature_names: List[str], title: str):
    fig, ax = _make_fig(size=(11, 10))
    im = ax.imshow(rollout_mat, cmap='magma', aspect='auto')
    n_show = min(len(feature_names), 20)
    step = max(1, len(feature_names) // n_show)
    tick_indices = list(range(0, len(feature_names), step))
    tick_labels = [feature_names[i][:10] for i in tick_indices]
    ax.set_xticks(tick_indices); ax.set_yticks(tick_indices)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(tick_labels, fontsize=9)
    ax.set_xlabel("To (Key)", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_ylabel("From (Query)", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_title(title, fontsize=PLOT.title_size, fontweight=PLOT.title_weight)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Rollout Weight", fontsize=PLOT.label_size)
    fig.tight_layout()
    return fig


def plot_pooling_weights_heatmap(class_feature_mat: np.ndarray, class_names: List[str], feature_names: List[str], title: str):
    fig, ax = _make_fig(size=(14, 6))
    im = ax.imshow(class_feature_mat, cmap='viridis', aspect='auto')
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_yticklabels(class_names, fontsize=10)
    n_show = min(len(feature_names), 25)
    step = max(1, len(feature_names) // n_show)
    tick_indices = list(range(0, len(feature_names), step))
    tick_labels = [feature_names[i][:10] for i in tick_indices]
    ax.set_xticks(tick_indices)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel("Features", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_ylabel("Classes", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_title(title, fontsize=PLOT.title_size, fontweight=PLOT.title_weight)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Avg Pool Weight", fontsize=PLOT.label_size)
    fig.tight_layout()
    return fig


def plot_attention_entropy_heatmap(ent: np.ndarray, feature_names: List[str], layer_idx: int):
    fig, ax = _make_fig(size=(12, 4.5))
    im = ax.imshow(ent, cmap='cividis', aspect='auto', vmin=0, vmax=1)
    ax.set_ylabel("Heads", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_xlabel("Query Tokens", fontsize=PLOT.label_size, fontweight=PLOT.label_weight)
    ax.set_title(f"Attention Entropy (Layer {layer_idx+1})", fontsize=PLOT.title_size, fontweight=PLOT.title_weight)

    n_show = min(len(feature_names), 25)
    step = max(1, len(feature_names) // n_show)
    tick_indices = list(range(0, len(feature_names), step))
    tick_labels = [feature_names[i][:10] for i in tick_indices]
    ax.set_xticks(tick_indices)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(np.arange(ent.shape[0]))
    ax.set_yticklabels([f"H{i+1}" for i in range(ent.shape[0])], fontsize=9)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Normalized Entropy", fontsize=PLOT.label_size)
    fig.tight_layout()
    return fig


# Utils
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def safe_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        _ = torch.arange(1, device="cuda") + 1
        return torch.device("cuda")
    except Exception:
        return torch.device("cpu")


def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def softmax_np(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=1, keepdims=True) + 1e-12)


def sync_device(device: torch.device):
    if device.type == "cuda":
        torch.cuda.synchronize()


class WallClock:
    def __init__(self, name: str, device: torch.device, store: Dict[str, float]):
        self.name = name
        self.device = device
        self.store = store
        self.t0 = None

    def __enter__(self):
        sync_device(self.device)
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        sync_device(self.device)
        t1 = time.perf_counter()
        self.store[self.name] = self.store.get(self.name, 0.0) + float(t1 - self.t0)


def count_parameters(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def bytes_of_parameters(model: nn.Module) -> int:
    total = 0
    for p in model.parameters():
        if p.requires_grad:
            total += p.numel() * 4
    return int(total)


def approx_transformer_time_complexity(n_tokens: int, d_model: int, d_ff: int, n_heads: int, n_layers: int) -> Dict[str, Any]:
    F_ = int(n_tokens)
    d = int(d_model)
    df = int(d_ff)
    L = int(n_layers)
    per_layer = f"O(F^2·d + F·d·d_ff)"
    total = f"O(L·(F^2·d + F·d·d_ff))"
    flops_attn_per_layer = (3 * F_ * d * d) + (2 * F_ * F_ * d) + (F_ * d * d)
    flops_ffn_per_layer = (2 * F_ * d * df)
    flops_per_layer = flops_attn_per_layer + flops_ffn_per_layer
    flops_total = L * flops_per_layer
    return {
        "F": F_, "d_model": d, "d_ff": df, "n_heads": int(n_heads), "n_layers": L,
        "bigO_per_layer": per_layer,
        "bigO_total": total,
        "approx_flops_per_layer": float(flops_per_layer),
        "approx_flops_total_per_sample": float(flops_total),
        "note": "Approximate FLOPs (matmul-dominant) for paper reporting."
    }


# ONE global stage-aware LR scheduler
def build_stage_aware_scheduler(
    optim: torch.optim.Optimizer,
    warmup_steps: int,
    total_steps: int,
    stage_step_boundaries: List[int],
    stage_lr_mults: List[float],
):
    assert len(stage_step_boundaries) == len(stage_lr_mults), "boundaries and mults must match"

    def stage_mult(step: int) -> float:
        for end_step, mult in zip(stage_step_boundaries, stage_lr_mults):
            if step < end_step:
                return float(mult)
        return float(stage_lr_mults[-1])

    def lr_lambda(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            base = float(step) / float(max(1, warmup_steps))
        else:
            denom = max(1, total_steps - warmup_steps)
            progress = float(step - warmup_steps) / float(denom)
            progress = min(max(progress, 0.0), 1.0)
            base = 0.5 * (1.0 + math.cos(math.pi * progress))
        return base * stage_mult(step)

    return torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)


# Config
@dataclass
class CFG:
    DATA_PATH: str = r"SKILICARSLAN_Anemia_DataSet.csv"
    OUT_ROOT: str = "Final_MTFTT_OUTPUTS"

    TARGET: str = "All_Class"
    FLAG_COLS: Tuple[str, ...] = ("HGB_Anemia_Class", "Iron_anemia_Class", "Folate_anemia_class", "B12_Anemia_class")

    RAW_FEATURES: Tuple[str, ...] = (
        "GENDER",
        "WBC", "NE", "LY", "MO", "EO", "BA",
        "RBC", "HGB", "HCT", "MCV", "MCH", "MCHC", "RDW",
        "PLT", "MPV", "PCT", "PDW",
        "SD", "SDTSD", "TSD",
        "FERRITTE", "FOLATE", "B12"
    )

    USE_SMOTE: bool = True
    SMOTE_K_MAX: int = 5

    NUM_CLASSES: int = 5
    CLASS_NAMES: Tuple[str, ...] = ("No_Anemia", "HGB_Anemia", "Iron_Def_Anemia", "Folate_Def_Anemia", "B12_Def_Anemia")

    SEED: int = 42
    TRAIN: float = 0.70
    VAL: float = 0.15
    TEST: float = 0.15
    CAL_FRAC_OF_VAL: float = 0.50

    D_MODEL: int = 256
    N_HEADS: int = 8
    N_LAYERS: int = 6
    D_FF: int = 1024
    DROPOUT: float = 0.20
    ATTN_DROPOUT: float = 0.15
    POOLING: str = "attn"

    

    LR: float = 2e-4
    WD: float = 1e-5
    CLIP_GRAD: float = 5.0
    BATCH: int = 256
    BATCH_EVAL: int = 512

    E_STAGE1: int = 30
    E_STAGE2: int = 40
    E_STAGE3: int = 80

    # CONTROL WHICH CLASSES ARE ACTIVE IN EACH STAGE (TRAIN + VAL)
    STAGE1_CLASSES: Tuple[int, ...] = (0, 1, 2)
    STAGE2_CLASSES: Tuple[int, ...] = (0, 1, 2, 4)    
    STAGE3_CLASSES: Tuple[int, ...] = (0, 1, 2, 3, 4)  

    RESET_OPT_EACH_STAGE: bool = False  # kept but ignored (we force one-optimizer)
    LR_STAGE1_MULT: float = 1.00
    LR_STAGE2_MULT: float = 0.75
    LR_STAGE3_MULT: float = 0.50

    LR_POLICY: str = "cosine"
    WARMUP_FRAC: float = 0.08

    PATIENCE: int = 12
    MIN_DELTA: float = 1e-4

    FOCAL_ALPHA: float = 0.20
    FOCAL_GAMMA: float = 1.80
    DICE_SMOOTH: float = 1e-6
    FOCAL_DICE_MIX: float = 0.5
    LABEL_SMOOTH: float = 0.02

    CLASS_WEIGHTING: str = "effective_num"
    CB_BETA: float = 0.9995

    AUG_NOISE_STD: float = 0.03

    OCC_SAMPLES: int = 256
    IG_SAMPLES: int = 256
    IG_STEPS: int = 64
    IG_BASELINE: str = "mean_train"

    ATTN_LAYER_VIZ: int = -1

    USE_CURRICULUM: bool = True
    USE_AUG: bool = True
    USE_FOCALDICE: bool = True
    USE_MULTITASK: bool = True
    USE_ATTN_REG: bool = True

    PROFILE_BATCHES: int = 8
    PROFILE_EVERY_N_EPOCHS: int = 5
    TIME_INFERENCE_REPEATS: int = 30

    SAVE_ALL_LAYERS_ATTENTION: bool = True
    ATTN_VIZ_MAX_BATCHES: int = 1
    SAVE_RAW_ATTN_MATRICES: bool = True

    LOG_STAGE2_VAL_SUBSET_AND_FULL: bool = True


cfg = CFG()
set_seed(cfg.SEED)
DEVICE = safe_device()

apply_plot_style(PLOT)
print("DEVICE:", DEVICE)

ensure_dir(Path(cfg.OUT_ROOT))
RUN_DIR = Path(cfg.OUT_ROOT) / time.strftime("V2.2Bin_MTFTT_run_%Y-%m-%d_at_%H-%M-%S")
ensure_dir(RUN_DIR)

PLOTS_DIR = RUN_DIR / "plots";  ensure_dir(PLOTS_DIR)
MODELS_DIR = RUN_DIR / "models"; ensure_dir(MODELS_DIR)
TABLES_DIR = RUN_DIR / "tables"; ensure_dir(TABLES_DIR)
METRICS_DIR = RUN_DIR / "metrics"; ensure_dir(METRICS_DIR)


#For Binary
BINARY_DIR = RUN_DIR / "binary"; ensure_dir(BINARY_DIR)
PLOTS_DIR = RUN_DIR / "plots";  ensure_dir(PLOTS_DIR)
MODELS_DIR = RUN_DIR / "models"; ensure_dir(MODELS_DIR)
TABLES_DIR = RUN_DIR / "tables";  ensure_dir(TABLES_DIR)
METRICS_DIR = RUN_DIR / "metrics"; ensure_dir(METRICS_DIR)
BINARY_DIR = RUN_DIR / "binary"; ensure_dir(BINARY_DIR)


with open(RUN_DIR / "config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2)

print("RUN_DIR:", RUN_DIR)

print("\n=== STAGE CLASS CONTROL (TRAIN + VAL) ===")
print("Stage-1 classes:", cfg.STAGE1_CLASSES, "=>", [cfg.CLASS_NAMES[i] for i in cfg.STAGE1_CLASSES])
print("Stage-2 classes:", cfg.STAGE2_CLASSES, "=>", [cfg.CLASS_NAMES[i] for i in cfg.STAGE2_CLASSES])
print("Stage-3 classes:", cfg.STAGE3_CLASSES, "=>", [cfg.CLASS_NAMES[i] for i in cfg.STAGE3_CLASSES])
print("========================================\n")


# Feature Engineering (AFTER SMOTE)
class FeatureEngineer:
    FERRITIN_REF_MALE = (40.0, 400.0)
    FERRITIN_REF_FEMALE = (15.0, 200.0)
    FOLATE_REF = (2.7, 17.0)
    B12_REF = (200.0, 900.0)
    MCHC_NORMAL = 34.0

    def names(self) -> List[str]:
        return [
            "Mentzer_Index",
            "NLR",
            "PLR",
            "RDW_MCV_Product",
            "RDW_to_RBC_Ratio",
            "MCHC_Deficit",
            "Ferritin_Gender_Norm",
            "B12_Folate_Combined",
        ]

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        d = df.copy()
        d["Mentzer_Index"] = d["MCV"] / (d["RBC"] + 1e-6)
        d["NLR"] = d["NE"] / (d["LY"] + 1e-6)
        d["PLR"] = d["PLT"] / (d["LY"] + 1e-6)
        d["RDW_MCV_Product"] = d["RDW"] * d["MCV"]
        d["RDW_to_RBC_Ratio"] = d["RDW"] / (d["RBC"] + 1e-6)
        d["MCHC_Deficit"] = self.MCHC_NORMAL - d["MCHC"]

        ferr_norm = np.where(
            d["GENDER"] == 1,  # verify encoding
            self._normalize_by_ref(d["FERRITTE"], *self.FERRITIN_REF_MALE),
            self._normalize_by_ref(d["FERRITTE"], *self.FERRITIN_REF_FEMALE),
        )
        d["Ferritin_Gender_Norm"] = ferr_norm

        fol_norm = self._normalize_by_ref(d["FOLATE"], *self.FOLATE_REF)
        b12_norm = self._normalize_by_ref(d["B12"], *self.B12_REF)
        d["B12_Folate_Combined"] = (fol_norm + b12_norm) / 2.0
        return d

    @staticmethod
    def _normalize_by_ref(values: pd.Series, ref_min: float, ref_max: float) -> np.ndarray:
        normalized = (values - ref_min) / (ref_max - ref_min + 1e-6)
        return np.clip(normalized, 0.0, 2.0)


FE = FeatureEngineer()


# Load + Validate
df0 = pd.read_csv(cfg.DATA_PATH, encoding="utf-8-sig")

missing = [c for c in (list(cfg.RAW_FEATURES) + [cfg.TARGET] + list(cfg.FLAG_COLS)) if c not in df0.columns]
assert not missing, f"Missing columns: {missing}"

labels = sorted(df0[cfg.TARGET].unique().tolist())
assert labels == [0, 1, 2, 3, 4], f"Unexpected labels in {cfg.TARGET}: {labels}"

flag = df0[list(cfg.FLAG_COLS)].values.astype(int)
y_all = df0[cfg.TARGET].values.astype(int)
assert flag[y_all == 0].sum() == 0, "Class 0 has non-zero subtype flags -> inconsistent dataset!"
flag_map = {1: 0, 2: 1, 3: 2, 4: 3}
for k, j in flag_map.items():
    rows = (y_all == k)
    assert flag[rows, j].sum() == rows.sum(), f"All_Class {k} mismatch with flag {cfg.FLAG_COLS[j]}"
    assert flag[rows].sum(axis=1).max() == 1, f"All_Class {k} rows have multiple subtype flags"
print("Label ↔ flags consistency OK")

assert df0[list(cfg.RAW_FEATURES) + [cfg.TARGET]].isna().sum().sum() == 0, "NaNs found in RAW/TARGET."

print("CSV shape:", df0.shape)
print("Class counts:\n", df0[cfg.TARGET].value_counts().sort_index())


# Splits (RAW first; no leakage)
train_df_raw, temp_df_raw = train_test_split(
    df0, test_size=(1.0 - cfg.TRAIN),
    random_state=cfg.SEED, stratify=df0[cfg.TARGET]
)
val_df_raw, test_df_raw = train_test_split(
    temp_df_raw, test_size=(cfg.TEST / (cfg.VAL + cfg.TEST)),
    random_state=cfg.SEED, stratify=temp_df_raw[cfg.TARGET]
)
val_eval_df_raw, cal_df_raw = train_test_split(
    val_df_raw, test_size=cfg.CAL_FRAC_OF_VAL,
    random_state=cfg.SEED, stratify=val_df_raw[cfg.TARGET]
)

print("\nSplit sizes (RAW):")
print("Train:", len(train_df_raw), "Val_eval:", len(val_eval_df_raw), "Cal:", len(cal_df_raw), "Test:", len(test_df_raw))


# SMOTE on RAW only (TRAIN only)
def smote_on_raw_train_only(train_df: pd.DataFrame, cfg_local: CFG) -> pd.DataFrame:
    X = train_df[list(cfg_local.RAW_FEATURES)].values.astype(np.float32)
    y = train_df[cfg_local.TARGET].values.astype(int)

    counts = np.bincount(y, minlength=cfg_local.NUM_CLASSES)
    min_count = int(counts.min())
    if min_count <= 1:
        raise ValueError(f"SMOTE cannot run: a class has <=1 sample. counts={counts.tolist()}")

    k = int(min(cfg_local.SMOTE_K_MAX, max(1, min_count - 1)))
    sm = SMOTE(random_state=cfg_local.SEED, k_neighbors=k)

    X_res, y_res = sm.fit_resample(X, y)

    df_res = pd.DataFrame(X_res, columns=list(cfg_local.RAW_FEATURES))
    df_res[cfg_local.TARGET] = y_res.astype(int)
    return df_res


if cfg.USE_SMOTE:
    train_df_smote_raw = smote_on_raw_train_only(train_df_raw, cfg)
    print("\n[SMOTE=ON] Train size before:", len(train_df_raw), "after:", len(train_df_smote_raw))
    print("[SMOTE=ON] Train class counts after:\n", train_df_smote_raw[cfg.TARGET].value_counts().sort_index())
else:
    train_df_smote_raw = train_df_raw.copy()
    print("\n[SMOTE=OFF] Train size:", len(train_df_smote_raw))


# FE after SMOTE
train_df = FE.apply(train_df_smote_raw)
val_eval_df = FE.apply(val_eval_df_raw)
cal_df = FE.apply(cal_df_raw)
test_df = FE.apply(test_df_raw)

ENGINEERED = FE.names()
FEATURES = list(cfg.RAW_FEATURES) + ENGINEERED
N_TOKENS = len(FEATURES)

for _df, _name in [(train_df, "train"), (val_eval_df, "val_eval"), (cal_df, "cal"), (test_df, "test")]:
    assert _df[FEATURES + [cfg.TARGET]].isna().sum().sum() == 0, f"NaNs found after FE in {_name}."

print("\nTotal tokens:", N_TOKENS)
print("Train/Val/Cal/Test shapes after FE:",
      train_df.shape, val_eval_df.shape, cal_df.shape, test_df.shape)


# StandardScaler AFTER FE (fit on TRAIN only)
scaler = StandardScaler()
Xtr = scaler.fit_transform(train_df[FEATURES].values)
Xva = scaler.transform(val_eval_df[FEATURES].values)
Xcal = scaler.transform(cal_df[FEATURES].values)
Xte = scaler.transform(test_df[FEATURES].values)

ytr = train_df[cfg.TARGET].values.astype(int)
yva = val_eval_df[cfg.TARGET].values.astype(int)
ycal = cal_df[cfg.TARGET].values.astype(int)
yte = test_df[cfg.TARGET].values.astype(int)

ytr_bin = (ytr > 0).astype(int)
yva_bin = (yva > 0).astype(int)
ycal_bin = (ycal > 0).astype(int)
yte_bin = (yte > 0).astype(int)

htr = train_df["HGB"].values.astype(np.float32)
hva = val_eval_df["HGB"].values.astype(np.float32)
hcal = cal_df["HGB"].values.astype(np.float32)
hte = test_df["HGB"].values.astype(np.float32)


# Conservative Medical Augmentation
class MedicalAugmentor:
    def __init__(self, feature_names: List[str], noise_std: float):
        self.idx = {n: i for i, n in enumerate(feature_names)}
        self.noise_std = float(noise_std)

    def _shift(self, z: np.ndarray, name: str, lo: float, hi: float):
        if name in self.idx:
            j = self.idx[name]
            z[j] += np.random.uniform(lo, hi)

    def augment(self, x_std: np.ndarray, cls: int) -> np.ndarray:
        z = x_std.copy()
        z += np.random.normal(0, self.noise_std, size=z.shape)
        if cls == 2:
            self._shift(z, "FERRITTE", -0.30, -0.08)
            self._shift(z, "MCV",      -0.20, -0.04)
        elif cls == 3:
            self._shift(z, "FOLATE",   -0.30, -0.08)
            self._shift(z, "MCV",       0.04,  0.20)
        elif cls == 4:
            self._shift(z, "B12",      -0.30, -0.08)
            self._shift(z, "MCV",       0.04,  0.20)
        return z


augmentor = MedicalAugmentor(FEATURES, noise_std=cfg.AUG_NOISE_STD)


# Dataset + Loaders
class AnemiaDS(Dataset):
    def __init__(self, X, y_sub, y_bin, y_hgb, stage: int, train: bool, enable_aug: bool):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_sub = torch.tensor(y_sub, dtype=torch.long)
        self.y_bin = torch.tensor(y_bin, dtype=torch.long)
        self.y_hgb = torch.tensor(y_hgb, dtype=torch.float32)
        self.stage = int(stage)
        self.train = bool(train)
        self.enable_aug = bool(enable_aug)

    def __len__(self): return self.X.shape[0]

    def __getitem__(self, i):
        x = self.X[i].clone()
        sub = int(self.y_sub[i].item())

        if self.train and self.enable_aug and self.stage >= 2 and sub in [2, 3, 4] and (np.random.rand() < 0.5):
            x = torch.tensor(augmentor.augment(x.numpy(), sub), dtype=torch.float32)

        return x, {"subtype": self.y_sub[i], "binary": self.y_bin[i], "hgb": self.y_hgb[i]}


def _filter_by_classes(X, y_sub, y_bin, y_hgb, keep_classes: Tuple[int, ...]):
    keep = np.isin(y_sub, list(keep_classes))
    return X[keep], y_sub[keep], y_bin[keep], y_hgb[keep]


def _stage_classes(cfg_local: CFG, stage: int) -> Tuple[int, ...]:
    if stage == 1:
        return cfg_local.STAGE1_CLASSES
    if stage == 2:
        return cfg_local.STAGE2_CLASSES
    return cfg_local.STAGE3_CLASSES


def make_train_loader(cfg_local, X, y_sub, y_bin, y_hgb, stage: int, enable_aug: bool) -> DataLoader:
    keep_classes = _stage_classes(cfg_local, int(stage))
    X2, y2, yb2, yh2 = _filter_by_classes(X, y_sub, y_bin, y_hgb, keep_classes)
    names = [cfg_local.CLASS_NAMES[i] for i in keep_classes]
    counts = np.bincount(y2, minlength=cfg_local.NUM_CLASSES)
    used_counts = {cfg_local.CLASS_NAMES[i]: int(counts[i]) for i in keep_classes}
    print(f"[TRAIN] Stage{stage} using classes={keep_classes} ({names}) | n={len(y2)} | counts={used_counts}")
    ds = AnemiaDS(X2, y2, yb2, yh2, stage=int(stage), train=True, enable_aug=enable_aug)
    return DataLoader(ds, batch_size=cfg_local.BATCH, shuffle=True, num_workers=0, drop_last=False)


def make_eval_loader(cfg_local, X, y_sub, y_bin, y_hgb) -> DataLoader:
    ds = AnemiaDS(X, y_sub, y_bin, y_hgb, stage=3, train=False, enable_aug=False)
    return DataLoader(ds, batch_size=cfg_local.BATCH_EVAL, shuffle=False, num_workers=0)


def make_eval_loader_for_classes(cfg_local, X, y_sub, y_bin, y_hgb, keep_classes: Optional[Tuple[int, ...]] = None, stage: int = 3) -> DataLoader:
    if keep_classes is not None:
        X, y_sub, y_bin, y_hgb = _filter_by_classes(X, y_sub, y_bin, y_hgb, keep_classes)
        names = [cfg_local.CLASS_NAMES[i] for i in keep_classes]
        counts = np.bincount(y_sub, minlength=cfg_local.NUM_CLASSES)
        used_counts = {cfg_local.CLASS_NAMES[i]: int(counts[i]) for i in keep_classes}
        print(f"[VAL ] Stage{stage} using classes={keep_classes} ({names}) | n={len(y_sub)} | counts={used_counts}")
    ds = AnemiaDS(X, y_sub, y_bin, y_hgb, stage=stage, train=False, enable_aug=False)
    return DataLoader(ds, batch_size=cfg_local.BATCH_EVAL, shuffle=False, num_workers=0)


# Build full and stage-specific VAL loaders (so we can swap per stage)
val_loader_full = make_eval_loader(cfg, Xva, yva, yva_bin, hva)
val_loader_s1 = make_eval_loader_for_classes(cfg, Xva, yva, yva_bin, hva, keep_classes=cfg.STAGE1_CLASSES, stage=1)
val_loader_s2 = make_eval_loader_for_classes(cfg, Xva, yva, yva_bin, hva, keep_classes=cfg.STAGE2_CLASSES, stage=2)
val_loader_s3 = make_eval_loader_for_classes(cfg, Xva, yva, yva_bin, hva, keep_classes=cfg.STAGE3_CLASSES, stage=3)

cal_loader = make_eval_loader(cfg, Xcal, ycal, ycal_bin, hcal)
test_loader = make_eval_loader(cfg, Xte, yte, yte_bin, hte)


# MTFTT Model
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, attn_dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=attn_dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.attn_w = None  # [B,H,F,F]

    def forward(self, x):
        out, w = self.mha(x, x, x, need_weights=True, average_attn_weights=False)
        self.attn_w = w.detach()
        x = self.norm1(x + self.drop(out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model, d_model)
        self.score = nn.Linear(d_model, 1)
        self.w = None  # [B,F]

    def forward(self, tokens):
        logits = self.score(torch.tanh(self.proj(tokens)))
        w = torch.softmax(logits, dim=1)
        self.w = w.detach().squeeze(-1)
        pooled = (tokens * w).sum(dim=1)
        return pooled


def build_attention_prior(feature_names: List[str], device: torch.device) -> torch.Tensor:
    high = {"HGB", "RBC", "MCV", "MCH", "MCHC", "RDW", "HCT", "FERRITTE", "FOLATE", "B12"}
    med  = {"Mentzer_Index", "B12_Folate_Combined", "MCHC_Deficit", "RDW_MCV_Product"}
    p = np.ones(len(feature_names), dtype=np.float32)
    for i, n in enumerate(feature_names):
        if n in high: p[i] = 3.0
        elif n in med: p[i] = 2.0
        else: p[i] = 1.0
    p = p / p.sum()
    return torch.tensor(p, dtype=torch.float32, device=device)


class MTFTT(nn.Module):
    def __init__(self, n_features, cfg_local: CFG):
        super().__init__()
        self.cfg = cfg_local
        self.n_features = n_features

        self.scalar_embed = nn.Linear(1, cfg_local.D_MODEL)
        self.id_embed = nn.Embedding(n_features, cfg_local.D_MODEL)
        self.register_buffer("feat_ids_buf", torch.arange(n_features).long())

        self.drop = nn.Dropout(cfg_local.DROPOUT)
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg_local.D_MODEL, cfg_local.N_HEADS, cfg_local.D_FF, cfg_local.DROPOUT, cfg_local.ATTN_DROPOUT)
            for _ in range(cfg_local.N_LAYERS)
        ])

        self.pool_attn = AttentionPooling(cfg_local.D_MODEL)
        self.pool_w = None

        self.head_sub = nn.Sequential(
            nn.Linear(cfg_local.D_MODEL, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, cfg_local.NUM_CLASSES)
        )
        self.head_bin = nn.Sequential(
            nn.Linear(cfg_local.D_MODEL, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
        self.head_hgb = nn.Sequential(
            nn.Linear(cfg_local.D_MODEL, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
        self.head_var = nn.Sequential(
            nn.Linear(cfg_local.D_MODEL, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Softplus()
        )

    def forward(self, x, return_attention: bool = False):
        B, F_ = x.shape
        feat_ids = self.feat_ids_buf.view(1, -1).expand(B, -1).to(x.device)

        tokens = self.scalar_embed(x.unsqueeze(-1)) + self.id_embed(feat_ids)
        tokens = self.drop(tokens)

        attn_layers = []
        for blk in self.blocks:
            tokens = blk(tokens)
            if return_attention:
                attn_layers.append(blk.attn_w)  # [B,H,F,F]

        if self.cfg.POOLING == "mean":
            pooled = tokens.mean(dim=1)
            pool_w_out = None
            self.pool_w = None
        else:
            pooled = self.pool_attn(tokens)
            pool_w_out = self.pool_attn.w
            self.pool_w = self.pool_attn.w

        out = {
            "subtype": self.head_sub(pooled),
            "binary": self.head_bin(pooled),
            "hgb": self.head_hgb(pooled).squeeze(-1),
            "var": self.head_var(pooled).squeeze(-1),
            "pool_w": pool_w_out,
        }
        if return_attention:
            out["attn_layers"] = attn_layers
        return out


# Losses
def weights_from_strategy(y_train: np.ndarray, cfg_local: CFG, device: torch.device) -> Optional[torch.Tensor]:
    if cfg_local.CLASS_WEIGHTING == "none":
        return None

    counts = np.bincount(y_train, minlength=cfg_local.NUM_CLASSES).astype(np.float32)
    if cfg_local.CLASS_WEIGHTING == "inv_freq":
        w = 1.0 / (counts + 1e-12)
        w = w * (cfg_local.NUM_CLASSES / (w.sum() + 1e-12))
        return torch.tensor(w, dtype=torch.float32, device=device)

    beta = float(cfg_local.CB_BETA)
    eff = (1.0 - np.power(beta, counts)) / (1.0 - beta + 1e-12)
    w = 1.0 / (eff + 1e-12)
    w = w * (cfg_local.NUM_CLASSES / (w.sum() + 1e-12))
    return torch.tensor(w, dtype=torch.float32, device=device)


class LabelSmoothingCE(nn.Module):
    def __init__(self, weight=None, smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.smoothing = float(smoothing)

    def forward(self, logits, target):
        if self.smoothing <= 0:
            return F.cross_entropy(logits, target, weight=self.weight)

        n_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)

        loss = torch.sum(-true_dist * log_probs, dim=1)
        if self.weight is None:
            return loss.mean()

        w = self.weight[target].detach()
        return (w * loss).mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, class_weights=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.class_weights = class_weights

    def forward(self, logits, y):
        p = torch.softmax(logits, dim=1)
        y_oh = F.one_hot(y, num_classes=logits.size(1)).float()
        pt = (p * y_oh).sum(dim=1).clamp_min(1e-8)
        ce = -torch.log(pt)
        fl = ((1 - pt) ** self.gamma) * ce

        if self.alpha is not None:
            alpha_t = self.alpha * y_oh + (1 - self.alpha) * (1 - y_oh)
            fl = alpha_t.sum(dim=1) * fl

        if self.class_weights is not None:
            fl = self.class_weights[y] * fl
        return fl.mean()


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = float(smooth)

    def forward(self, logits, y):
        p = torch.softmax(logits, dim=1)
        y_oh = F.one_hot(y, num_classes=logits.size(1)).float()
        inter = (p * y_oh).sum(dim=0)
        card = (p + y_oh).sum(dim=0)
        dice = (2 * inter + self.smooth) / (card + self.smooth)
        return 1 - dice.mean()


class FocalDice(nn.Module):
    def __init__(self, mix=0.5, alpha=0.25, gamma=2.0, smooth=1e-6, class_weights=None):
        super().__init__()
        self.mix = float(mix)
        self.focal = FocalLoss(alpha=alpha, gamma=gamma, class_weights=class_weights)
        self.dice = DiceLoss(smooth=smooth)

    def forward(self, logits, y):
        return (1 - self.mix) * self.focal(logits, y) + self.mix * self.dice(logits, y)


class UncertaintyTaskWeights(nn.Module):
    def __init__(self, tasks: List[str]):
        super().__init__()
        self.tasks = tasks
        self.log_vars = nn.ParameterDict({t: nn.Parameter(torch.zeros(1)) for t in tasks})

    def weight_terms(self, losses: Dict[str, torch.Tensor]) -> torch.Tensor:
        total = torch.tensor(0.0, device=next(self.parameters()).device)
        for t in self.tasks:
            Li = losses[t]
            s = self.log_vars[t]
            total = total + torch.exp(-s) * Li + s
        return total

    @torch.no_grad()
    def snapshot(self) -> Dict[str, float]:
        out = {}
        for t in self.tasks:
            out[f"log_var_{t}"] = float(self.log_vars[t].item())
            out[f"w_{t}"] = float(torch.exp(-self.log_vars[t]).item())
        return out


class MultiTaskLoss(nn.Module):
    def __init__(self, cfg_local: CFG, class_weights: Optional[torch.Tensor], attn_prior: torch.Tensor,
                 use_focaldice=True, use_multitask=True, use_attn_reg=True):
        super().__init__()
        self.cfg = cfg_local
        self.use_focaldice = bool(use_focaldice)
        self.use_multitask = bool(use_multitask)
        self.use_attn_reg = bool(use_attn_reg)

        if self.use_focaldice:
            self.loss_sub = FocalDice(
                mix=cfg_local.FOCAL_DICE_MIX,
                alpha=cfg_local.FOCAL_ALPHA,
                gamma=cfg_local.FOCAL_GAMMA,
                smooth=cfg_local.DICE_SMOOTH,
                class_weights=class_weights
            )
        else:
            self.loss_sub = LabelSmoothingCE(weight=class_weights, smoothing=cfg_local.LABEL_SMOOTH)

        self.loss_bin = LabelSmoothingCE(weight=None, smoothing=cfg_local.LABEL_SMOOTH)
        self.attn_prior = attn_prior

        tasks = ["subtype"]
        if self.use_multitask:
            tasks += ["binary", "reg"]
            if self.use_attn_reg:
                tasks += ["attn"]
        self.autoW = UncertaintyTaskWeights(tasks)

    def forward(self, out, y):
        L_sub = self.loss_sub(out["subtype"], y["subtype"])

        if not self.use_multitask:
            z = torch.tensor(0.0, device=L_sub.device)
            return {"total": L_sub, "subtype": L_sub, "binary": z, "reg": z, "attn": z, "auto": self.autoW.snapshot()}

        L_bin = self.loss_bin(out["binary"], y["binary"])

        var = out["var"].clamp_min(1e-6)
        logvar = torch.log(var)
        mse = (out["hgb"] - y["hgb"]) ** 2
        L_reg = (torch.exp(-logvar) * mse + logvar).mean()

        if self.use_attn_reg and (out["pool_w"] is not None):
            L_attn = ((out["pool_w"] - self.attn_prior) ** 2).mean()
        else:
            L_attn = torch.tensor(0.0, device=L_sub.device)

        parts = {"subtype": L_sub, "binary": L_bin, "reg": L_reg}
        if self.use_attn_reg:
            parts["attn"] = L_attn

        total = self.autoW.weight_terms(parts)
        return {"total": total, "subtype": L_sub, "binary": L_bin, "reg": L_reg, "attn": L_attn, "auto": self.autoW.snapshot()}


# Calibration
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_T = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        T = torch.exp(self.log_T).clamp_min(1e-3)
        return logits / T

    def temperature(self):
        return float(torch.exp(self.log_T).item())


@torch.no_grad()
def collect_logits_targets(model, loader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_y = [], []
    for X, y in loader:
        X = X.to(DEVICE)
        out = model(X, return_attention=False)
        all_logits.append(out["subtype"].detach().cpu().numpy())
        all_y.append(y["subtype"].cpu().numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)


def fit_temperature(model, cal_loader, max_iter=200, lr=0.05, timing_store: Optional[Dict[str, float]] = None) -> TemperatureScaler:
    timing_store = timing_store if timing_store is not None else {}
    with WallClock("calibration_temperature_fit_sec", DEVICE, timing_store):
        logits_np, y_np = collect_logits_targets(model, cal_loader)
        logits = torch.tensor(logits_np, dtype=torch.float32, device=DEVICE)
        y = torch.tensor(y_np, dtype=torch.long, device=DEVICE)

        scalerT = TemperatureScaler().to(DEVICE)
        opt = torch.optim.LBFGS(scalerT.parameters(), lr=lr, max_iter=max_iter)
        nll = nn.CrossEntropyLoss()

        def closure():
            opt.zero_grad(set_to_none=True)
            loss = nll(scalerT(logits), y)
            loss.backward()
            return loss

        opt.step(closure)
    return scalerT


def expected_calibration_error(probs: np.ndarray, y: np.ndarray, n_bins: int = 15) -> float:
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc = (pred == y).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.mean()) * abs(acc[mask].mean() - conf[mask].mean())
    return float(ece)


def multiclass_brier(probs: np.ndarray, y: np.ndarray, num_classes: int) -> float:
    y_oh = np.eye(num_classes)[y]
    return float(np.mean(np.sum((probs - y_oh) ** 2, axis=1)))


# XAI CORE
@torch.no_grad()
def attention_rollout(attn_layers: List[torch.Tensor]) -> np.ndarray:
    mats = [a.mean(dim=1) for a in attn_layers]  # [B,F,F]
    B, F_, _ = mats[0].shape
    eye = torch.eye(F_, device=mats[0].device).unsqueeze(0).expand(B, -1, -1)
    mats = [(m + eye) / 2.0 for m in mats]
    mats = [m / (m.sum(dim=-1, keepdims=True) + 1e-8) for m in mats]
    R = mats[0]
    for m in mats[1:]:
        R = torch.bmm(m, R)
    return R.mean(dim=0).detach().cpu().numpy()


@torch.no_grad()
def compute_global_attention_importance(model, loader, n_tokens: int, max_batches=30) -> np.ndarray:
    model.eval()
    w_sum, n = None, 0
    for i, (X, _) in enumerate(loader):
        if i >= max_batches:
            break
        X = X.to(DEVICE)
        out = model(X, return_attention=False)
        if out["pool_w"] is None:
            w = np.ones((X.size(0), n_tokens), dtype=np.float32) / float(n_tokens)
        else:
            w = out["pool_w"].detach().cpu().numpy()
        w_sum = w.sum(axis=0) if w_sum is None else (w_sum + w.sum(axis=0))
        n += w.shape[0]
    return w_sum / max(n, 1)


@torch.no_grad()
def per_class_attention_importance(model, loader, n_tokens: int, num_classes=5, max_batches=60) -> Dict[int, np.ndarray]:
    model.eval()
    sums = {c: None for c in range(num_classes)}
    counts = {c: 0 for c in range(num_classes)}
    for i, (X, _) in enumerate(loader):
        if i >= max_batches:
            break
        X = X.to(DEVICE)
        out = model(X, return_attention=False)
        pred = out["subtype"].argmax(dim=1).detach().cpu().numpy()

        if out["pool_w"] is None:
            w = np.ones((X.size(0), n_tokens), dtype=np.float32) / float(n_tokens)
        else:
            w = out["pool_w"].detach().cpu().numpy()

        for c in range(num_classes):
            m = (pred == c)
            if m.sum() == 0:
                continue
            add = w[m].sum(axis=0)
            sums[c] = add if sums[c] is None else (sums[c] + add)
            counts[c] += int(m.sum())
    return {c: (np.zeros(n_tokens, np.float32) if sums[c] is None else sums[c] / max(counts[c], 1))
            for c in range(num_classes)}


@torch.no_grad()
def occlusion_sensitivity(model, X_np: np.ndarray, y_true: np.ndarray, baseline: np.ndarray,
                          n_tokens: int, n_samples: int) -> np.ndarray:
    model.eval()
    idx = np.random.choice(len(X_np), size=min(n_samples, len(X_np)), replace=False)
    X = torch.tensor(X_np[idx], dtype=torch.float32, device=DEVICE)
    y = torch.tensor(y_true[idx], dtype=torch.long, device=DEVICE)

    out0 = model(X)
    p0 = torch.softmax(out0["subtype"], dim=1)
    p0_true = p0[torch.arange(p0.size(0), device=DEVICE), y].detach()

    imp = np.zeros(n_tokens, dtype=np.float32)
    for j in range(n_tokens):
        Xocc = X.clone()
        Xocc[:, j] = float(baseline[j])
        out = model(Xocc)
        p = torch.softmax(out["subtype"], dim=1)
        p_true = p[torch.arange(p.size(0), device=DEVICE), y].detach()
        imp[j] = float((p0_true - p_true).mean().item())
    return imp


def attention_entropy_from_layer(attn_bhff: torch.Tensor) -> np.ndarray:
    a = attn_bhff.mean(dim=0)  # [H,F,F]
    a = a.clamp_min(1e-12)
    ent = -(a * torch.log(a)).sum(dim=-1)  # [H,F]
    ent = ent / math.log(a.shape[-1] + 1e-12)
    return ent.detach().cpu().numpy().clip(0.0, 1.0)


def integrated_gradients(model: nn.Module, x: torch.Tensor, target_class: int,
                         baseline: torch.Tensor, steps: int = 64) -> torch.Tensor:
    model.eval()
    x = x.detach()
    baseline = baseline.detach()
    assert x.ndim == 1 and baseline.ndim == 1 and x.shape == baseline.shape

    with torch.enable_grad():
        alphas = torch.linspace(0.0, 1.0, steps + 1, device=x.device).view(-1, 1)
        xs = baseline.unsqueeze(0) + alphas * (x.unsqueeze(0) - baseline.unsqueeze(0))
        xs.requires_grad_(True)

        out = model(xs, return_attention=False)
        logits = out["subtype"]
        score = logits[:, int(target_class)].sum()

        grads = torch.autograd.grad(score, xs, create_graph=False, retain_graph=False)[0]
        avg_grads = 0.5 * (grads[:-1] + grads[1:])
        ig = (x - baseline) * avg_grads.mean(dim=0)

    return ig.detach()


def compute_ig_pack(model: nn.Module, X_np: np.ndarray, y_true: np.ndarray,
                    baseline: np.ndarray, n_samples: int, steps: int
                   ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    n = min(int(n_samples), len(X_np))
    idx = np.random.choice(len(X_np), size=n, replace=False)

    baseline_t = torch.tensor(baseline, dtype=torch.float32, device=DEVICE)

    sample_pick = int(idx[0])
    sample_true = int(y_true[sample_pick])

    with torch.no_grad():
        Xs = torch.tensor(X_np[sample_pick], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        sample_pred = int(model(Xs, return_attention=False)["subtype"].argmax(dim=1).item())

    all_attr = []
    sample_attr = None

    for i in idx:
        x = torch.tensor(X_np[i], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            pred = int(model(x.unsqueeze(0), return_attention=False)["subtype"].argmax(dim=1).item())
        attr = integrated_gradients(model, x, target_class=pred, baseline=baseline_t, steps=int(steps))
        attr_np = attr.detach().cpu().numpy()

        all_attr.append(attr_np)
        if int(i) == sample_pick:
            sample_attr = attr_np

    all_attr = np.stack(all_attr, axis=0)
    if sample_attr is None:
        sample_attr = all_attr[0]

    sample_meta = np.array([sample_pick, sample_true, sample_pred], dtype=int)
    return all_attr, sample_attr, sample_meta


# Paper metrics pack
def per_class_from_cm(cm: np.ndarray, class_names: List[str]) -> pd.DataFrame:
    total = cm.sum()
    rows = []
    for i, name in enumerate(class_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = total - TP - FN - FP

        sens = TP / (TP + FN + 1e-12)
        spec = TN / (TN + FP + 1e-12)
        ppv  = TP / (TP + FP + 1e-12)
        npv  = TN / (TN + FN + 1e-12)

        rows.append({
            "class": name,
            "TP": int(TP), "FP": int(FP), "FN": int(FN), "TN": int(TN),
            "sensitivity_recall": float(sens),
            "specificity": float(spec),
            "precision_ppv": float(ppv),
            "npv": float(npv),
        })
    return pd.DataFrame(rows)


def compute_full_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_pred))
    out["macro_precision"] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    out["macro_recall"] = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    out["weighted_precision"] = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    out["weighted_recall"] = float(recall_score(y_true, y_pred, average="weighted", zero_division=0))
    out["weighted_f1"] = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    out["micro_precision"] = float(precision_score(y_true, y_pred, average="micro", zero_division=0))
    out["micro_recall"] = float(recall_score(y_true, y_pred, average="micro", zero_division=0))
    out["micro_f1"] = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    out["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    out["kappa"] = float(cohen_kappa_score(y_true, y_pred))
    return out


# Decision Curve Analysis (binary anemia)
def decision_curve_binary(y_true_bin: np.ndarray, p_pos: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    y_true_bin = y_true_bin.astype(int)
    p_pos = np.clip(p_pos, 1e-8, 1 - 1e-8)
    N = len(y_true_bin)
    prev = y_true_bin.mean()

    rows = []
    for pt in thresholds:
        pred = (p_pos >= pt).astype(int)
        TP = ((pred == 1) & (y_true_bin == 1)).sum()
        FP = ((pred == 1) & (y_true_bin == 0)).sum()

        w = pt / (1 - pt)
        nb_model = (TP / N) - (FP / N) * w
        nb_all = prev - (1 - prev) * w
        nb_none = 0.0

        rows.append({
            "threshold": float(pt),
            "net_benefit_model": float(nb_model),
            "net_benefit_treat_all": float(nb_all),
            "net_benefit_treat_none": float(nb_none),
        })
    return pd.DataFrame(rows)


# Eval helpers
@torch.no_grad()
def eval_epoch(model, loader, loss_fn, timing_store: Optional[Dict[str, float]] = None, timing_key: str = "eval") -> Dict:
    timing_store = timing_store if timing_store is not None else {}
    model.eval()
    agg = {"total": 0.0, "subtype": 0.0, "binary": 0.0, "reg": 0.0, "attn": 0.0}
    y_true, y_pred, logits_all = [], [], []
    n = 0

    with WallClock(f"{timing_key}_sec", DEVICE, timing_store):
        for X, y in loader:
            X = X.to(DEVICE)
            y = {k: v.to(DEVICE) for k, v in y.items()}
            out = model(X)
            L = loss_fn(out, y)
            for k in agg:
                agg[k] += float(L[k].item())

            logits_all.append(out["subtype"].detach().cpu().numpy())
            pred = out["subtype"].argmax(dim=1).detach().cpu().numpy()

            y_true.append(y["subtype"].detach().cpu().numpy())
            y_pred.append(pred)
            n += int(X.size(0))

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    logits_all = np.concatenate(logits_all)

    for k in agg:
        agg[k] /= max(len(loader), 1)

    agg.update(compute_full_metrics(y_true, y_pred))
    agg["y_true"], agg["y_pred"], agg["logits"] = y_true, y_pred, logits_all
    agg[f"{timing_key}_samples"] = int(n)
    agg[f"{timing_key}_throughput_sps"] = float(n / max(timing_store.get(f"{timing_key}_sec", 1e-12), 1e-12))
    return agg



# train_one_epoch uses global_step and global scheduler (no resets)
def train_one_epoch(
    model, loader, optim, scaler_amp, loss_fn, clip_grad: float,
    use_amp: bool, sched=None, global_step_start: int = 0,
    timing_store: Optional[Dict[str, float]] = None,
    profile_batches: int = 0,
    profile_this_epoch: bool = False,
) -> Tuple[Dict, int, Dict, Dict]:
    timing_store = timing_store if timing_store is not None else {}
    model.train()
    agg = {"total": 0.0, "subtype": 0.0, "binary": 0.0, "reg": 0.0, "attn": 0.0}
    global_step = int(global_step_start)
    last_auto = {}

    y_true_all, y_pred_all = [], []
    correct, n = 0, 0

    fw_times, bw_times, step_times = [], [], []

    with WallClock("train_epoch_sec", DEVICE, timing_store):
        pbar = tqdm(loader, leave=False)
        for bidx, (X, y) in enumerate(pbar):
            X = X.to(DEVICE)
            y = {k: v.to(DEVICE) for k, v in y.items()}
            optim.zero_grad(set_to_none=True)

            do_profile = (profile_this_epoch and (profile_batches > 0) and (bidx < profile_batches))
            if do_profile:
                sync_device(DEVICE); t_step0 = time.perf_counter()
                sync_device(DEVICE); t_fw0 = time.perf_counter()

            if use_amp:
                with torch.amp.autocast(device_type=DEVICE.type, enabled=True):
                    out = model(X)
                    L = loss_fn(out, y)
            else:
                out = model(X)
                L = loss_fn(out, y)

            if do_profile:
                sync_device(DEVICE); t_fw1 = time.perf_counter()
                fw_times.append(1000.0 * (t_fw1 - t_fw0))
                sync_device(DEVICE); t_bw0 = time.perf_counter()

            if use_amp:
                scaler_amp.scale(L["total"]).backward()
                scaler_amp.unscale_(optim)
                if clip_grad and clip_grad > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                scaler_amp.step(optim)
                scaler_amp.update()
            else:
                L["total"].backward()
                if clip_grad and clip_grad > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                optim.step()

            if do_profile:
                sync_device(DEVICE); t_bw1 = time.perf_counter()
                bw_times.append(1000.0 * (t_bw1 - t_bw0))

            # global scheduler step (continuous)
            if sched is not None:
                sched.step()

            if do_profile:
                sync_device(DEVICE); t_step1 = time.perf_counter()
                step_times.append(1000.0 * (t_step1 - t_step0))

            global_step += 1

            for k in agg:
                agg[k] += float(L[k].item())

            pred = out["subtype"].argmax(dim=1)
            correct += int((pred == y["subtype"]).sum().item())
            n += X.size(0)

            y_true_all.append(y["subtype"].detach().cpu().numpy())
            y_pred_all.append(pred.detach().cpu().numpy())

            last_auto = L.get("auto", {})
            pbar.set_postfix(loss=f"{L['total'].item():.4f}", acc=f"{100 * correct / n:.2f}%")

    for k in agg:
        agg[k] /= max(len(loader), 1)

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    agg["train_acc_pct"] = 100.0 * correct / max(n, 1)
    agg["train_macro_f1"] = float(f1_score(y_true_all, y_pred_all, average="macro", zero_division=0))
    agg["train_samples"] = int(n)
    agg["train_throughput_sps"] = float(n / max(timing_store.get("train_epoch_sec", 1e-12), 1e-12))

    batch_profile = {}
    if len(step_times) > 0:
        batch_profile = {
            "profile_batches": int(len(step_times)),
            "fw_ms_mean": float(np.mean(fw_times)),
            "bw_ms_mean": float(np.mean(bw_times)),
            "step_ms_mean": float(np.mean(step_times)),
        }
    return agg, global_step, last_auto, batch_profile


def reliability_bins(probs: np.ndarray, y: np.ndarray, n_bins=15):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_conf, bin_acc = [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        bin_conf.append(conf[m].mean())
        bin_acc.append(correct[m].mean())
    return np.array(bin_conf), np.array(bin_acc)


def plot_auto_weights(hist_df: pd.DataFrame, tag: str, stage_boundaries: Optional[List[int]]):
    cols = [c for c in hist_df.columns if c.startswith("w_")]
    if len(cols) == 0:
        return None
    fig, ax = _make_fig()
    for c in cols:
        ax.plot(hist_df["epoch_global"], hist_df[c], linewidth=PLOT.line_width, label=c)
    if stage_boundaries:
        for b in stage_boundaries:
            ax.axvline(b, linestyle="--", linewidth=PLOT.vline_width, alpha=0.65)
    _bold_title_labels(ax, f"{tag}: Learned Automatic Task Weights (exp(-log_var))",
                       "Global epoch", "Weight multiplier")
    ax.legend(fontsize=PLOT.legend_size, ncols=2, frameon=PLOT.legend_frame, loc=PLOT.legend_loc)
    if PLOT.bold_all_text:
        for t in ax.get_legend().get_texts():
            t.set_fontweight(PLOT.legend_weight)
    style_axes(ax)
    fig.tight_layout()
    return fig


# Experiment runner (FINAL FIXED + Stage2 excludes class-4)
def run_proposed_model(
    tag: str,
    overrides: Optional[Dict[str, Any]] = None,
    use_curriculum: bool = True,
    use_aug: bool = True,
    use_focaldice: bool = True,
    use_multitask: bool = True,
    use_attn_reg: bool = True,
    epoch_scale: float = 1.0
) -> Dict:
    overrides = overrides or {}

    cfg_local = replace(cfg)
    for k, v in overrides.items():
        if hasattr(cfg_local, k):
            setattr(cfg_local, k, v)

    e1 = max(1, int(cfg_local.E_STAGE1 * epoch_scale))
    e2 = max(1, int(cfg_local.E_STAGE2 * epoch_scale))
    e3 = max(1, int(cfg_local.E_STAGE3 * epoch_scale))

    timing: Dict[str, float] = {}
    timing_rows = []
    batch_profile_rows = []

    summary = None
    run_pdf = None

    
    # ALL RUN WORK
    with WallClock("total_run_wallclock_sec", DEVICE, timing):

        model = MTFTT(n_features=N_TOKENS, cfg_local=cfg_local).to(DEVICE)
        class_w = weights_from_strategy(ytr, cfg_local, DEVICE)
        attn_prior = build_attention_prior(FEATURES, DEVICE)

        loss_fn = MultiTaskLoss(
            cfg_local, class_w, attn_prior,
            use_focaldice=use_focaldice,
            use_multitask=use_multitask,
            use_attn_reg=use_attn_reg
        ).to(DEVICE)

        n_params = count_parameters(model) + count_parameters(loss_fn.autoW)
        param_bytes = bytes_of_parameters(model) + bytes_of_parameters(loss_fn.autoW)
        complexity = approx_transformer_time_complexity(
            n_tokens=N_TOKENS, d_model=cfg_local.D_MODEL, d_ff=cfg_local.D_FF,
            n_heads=cfg_local.N_HEADS, n_layers=cfg_local.N_LAYERS
        )

        with open(METRICS_DIR / f"{tag}_complexity.json", "w") as f:
            json.dump({
                "n_tokens": N_TOKENS,
                "n_params_trainable_model_plus_autoweights": int(n_params),
                "approx_param_memory_bytes_fp32": int(param_bytes),
                "complexity": complexity
            }, f, indent=2)

        use_amp = (DEVICE.type == "cuda")
        scaler_amp = torch.amp.GradScaler(enabled=use_amp)

        # ONE optimizer for whole run (model + autoW)
        params = list(model.parameters()) + list(loss_fn.autoW.parameters())
        optim = torch.optim.AdamW(params, lr=float(cfg_local.LR), weight_decay=float(cfg_local.WD))

        # Build one global scheduler across stages
        tmp_loader_s1 = make_train_loader(cfg_local, Xtr, ytr, ytr_bin, htr, stage=1, enable_aug=use_aug)
        tmp_loader_s2 = make_train_loader(cfg_local, Xtr, ytr, ytr_bin, htr, stage=2, enable_aug=use_aug)
        tmp_loader_s3 = make_train_loader(cfg_local, Xtr, ytr, ytr_bin, htr, stage=3, enable_aug=use_aug)
        steps_s1 = len(tmp_loader_s1)
        steps_s2 = len(tmp_loader_s2)
        steps_s3 = len(tmp_loader_s3)

        total_steps = steps_s1 * e1 + steps_s2 * e2 + steps_s3 * e3
        warmup_steps = int(cfg_local.WARMUP_FRAC * total_steps) if cfg_local.LR_POLICY != "constant" else 0

        end_s1 = steps_s1 * e1
        end_s2 = end_s1 + steps_s2 * e2
        end_s3 = end_s2 + steps_s3 * e3
        stage_step_boundaries = [end_s1, end_s2, end_s3]
        stage_lr_mults = [cfg_local.LR_STAGE1_MULT, cfg_local.LR_STAGE2_MULT, cfg_local.LR_STAGE3_MULT]

        sched = build_stage_aware_scheduler(
            optim,
            warmup_steps=warmup_steps,
            total_steps=max(1, total_steps),
            stage_step_boundaries=stage_step_boundaries,
            stage_lr_mults=stage_lr_mults
        )

        history = []
        stage_boundaries = []
        global_epoch = 0
        global_step = 0  # never reset

        best_va = -1.0
        best_state = None
        best_loss_state = None
        patience = 0

        def save_best_state():
            nonlocal best_state, best_loss_state
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_loss_state = {k: v.detach().cpu().clone() for k, v in loss_fn.state_dict().items()}

        def load_best_state():
            if best_state is not None:
                model.load_state_dict(best_state)
            if best_loss_state is not None:
                loss_fn.load_state_dict(best_loss_state)

        # stage-aware validation loader is consistent with stage classes
        def get_stage_val_loader(stage: int):
            if not use_curriculum:
                return val_loader_s3
            if stage == 1:
                return val_loader_s1
            if stage == 2:
                return val_loader_s2
            return val_loader_s3

        def do_stage(stage: int, epochs: int, allow_early_stop: bool):
            nonlocal global_epoch, global_step, patience, best_va

            with WallClock(f"stage_{stage}_total_sec", DEVICE, timing):
                tr_loader = make_train_loader(cfg_local, Xtr, ytr, ytr_bin, htr, stage=stage, enable_aug=use_aug)
                stage_val_loader = get_stage_val_loader(stage)

                for ep in range(1, epochs + 1):
                    global_epoch += 1
                    profile_this_epoch = (cfg_local.PROFILE_BATCHES > 0) and (global_epoch % cfg_local.PROFILE_EVERY_N_EPOCHS == 0)

                    epoch_timing = {}
                    tr, global_step, auto_snap, batch_prof = train_one_epoch(
                        model, tr_loader, optim, scaler_amp, loss_fn,
                        clip_grad=cfg_local.CLIP_GRAD,
                        use_amp=use_amp,
                        sched=sched,
                        global_step_start=global_step,
                        timing_store=epoch_timing,
                        profile_batches=cfg_local.PROFILE_BATCHES,
                        profile_this_epoch=profile_this_epoch
                    )

                    va_timing = {}
                    va = eval_epoch(model, stage_val_loader, loss_fn, timing_store=va_timing, timing_key="val_eval")

                    # Optional Stage-2 logging both subset and "full" validation
                    # Here "subset" = Stage-2 classes; "full" = Stage-3 classes (all)
                    va_s2_subset = None
                    va_s2_full = None
                    if use_curriculum and (stage == 2) and cfg_local.LOG_STAGE2_VAL_SUBSET_AND_FULL:
                        _t1 = {}
                        va_s2_subset = eval_epoch(model, val_loader_s2, loss_fn, timing_store=_t1, timing_key="val_s2_subset")
                        _t2 = {}
                        va_s2_full = eval_epoch(model, val_loader_s3, loss_fn, timing_store=_t2, timing_key="val_s2_full")

                    rec = {
                        "tag": tag,
                        "stage": stage,
                        "epoch_in_stage": ep,
                        "epoch_global": global_epoch,
                        "lr": float(optim.param_groups[0]["lr"]),

                        "tr_total": tr["total"], "va_total": va["total"],
                        "tr_sub": tr["subtype"], "va_sub": va["subtype"],
                        "tr_bin": tr["binary"], "va_bin": va["binary"],
                        "tr_reg": tr["reg"], "va_reg": va["reg"],
                        "tr_attn": tr["attn"], "va_attn": va["attn"],

                        "tr_acc_pct": tr["train_acc_pct"],
                        "tr_macro_f1": tr["train_macro_f1"],

                        "va_acc": va["acc"],
                        "va_bal_acc": va["bal_acc"],
                        "va_macro_f1": va["macro_f1"],
                        "va_weighted_f1": va["weighted_f1"],
                        "va_macro_precision": va["macro_precision"],
                        "va_macro_recall": va["macro_recall"],
                        "va_mcc": va["mcc"],
                        "va_kappa": va["kappa"],
                    }

                    if va_s2_subset is not None:
                        rec["va_s2_subset_macro_f1"] = va_s2_subset["macro_f1"]
                        rec["va_s2_subset_bal_acc"] = va_s2_subset["bal_acc"]
                    if va_s2_full is not None:
                        rec["va_s2_full_macro_f1"] = va_s2_full["macro_f1"]
                        rec["va_s2_full_bal_acc"] = va_s2_full["bal_acc"]

                    for k, v in auto_snap.items():
                        rec[k] = v
                    history.append(rec)

                    timing_rows.append({
                        "tag": tag,
                        "stage": int(stage),
                        "epoch_in_stage": int(ep),
                        "epoch_global": int(global_epoch),
                        "train_epoch_sec": float(epoch_timing.get("train_epoch_sec", np.nan)),
                        "train_throughput_sps": float(tr.get("train_throughput_sps", np.nan)),
                        "val_eval_sec": float(va_timing.get("val_eval_sec", np.nan)),
                        "val_throughput_sps": float(va.get("val_eval_throughput_sps", np.nan)),
                        "lr": float(optim.param_groups[0]["lr"]),
                        "device": str(DEVICE),
                        "amp": bool(use_amp),
                    })

                    if batch_prof:
                        batch_profile_rows.append({
                            "tag": tag,
                            "stage": int(stage),
                            "epoch_global": int(global_epoch),
                            **batch_prof
                        })

                    if ep % 5 == 0:
                        print(f"[{tag}] Stage{stage} Ep{ep} | val_macroF1={va['macro_f1']:.4f} "
                              f"bal_acc={va['bal_acc']:.4f} val_loss={va['total']:.4f} lr={optim.param_groups[0]['lr']:.2e}")

                    # Early stop only in Stage-3 and on Stage-3 validation
                    if allow_early_stop:
                        improved = (va["macro_f1"] > best_va + cfg_local.MIN_DELTA)
                        if improved:
                            best_va = va["macro_f1"]
                            save_best_state()
                            patience = 0
                        else:
                            patience += 1
                        if patience >= cfg_local.PATIENCE:
                            print(f"[{tag}] Early stopping triggered at Stage3 Ep{ep} (patience={cfg_local.PATIENCE})")
                            break

                stage_boundaries.append(global_epoch)

        if use_curriculum:
            print(f"\n=== {tag}: STAGE 1 (classes={cfg_local.STAGE1_CLASSES}) ===")
            do_stage(stage=1, epochs=e1, allow_early_stop=False)

            print(f"\n=== {tag}: STAGE 2 (classes={cfg_local.STAGE2_CLASSES}) ===")
            do_stage(stage=2, epochs=e2, allow_early_stop=False)

            print(f"\n=== {tag}: STAGE 3 (classes={cfg_local.STAGE3_CLASSES}) ===")
            do_stage(stage=3, epochs=e3, allow_early_stop=True)
        else:
            print(f"\n=== {tag}: NO CURRICULUM (single stage uses STAGE3_CLASSES) ===")
            do_stage(stage=3, epochs=e1 + e2 + e3, allow_early_stop=True)

        load_best_state()

        te_timing = {}
        te = eval_epoch(model, test_loader, loss_fn, timing_store=te_timing, timing_key="test_eval")
        timing.update(te_timing)
        cm = confusion_matrix(te["y_true"], te["y_pred"])
        rep = classification_report(te["y_true"], te["y_pred"], target_names=list(cfg_local.CLASS_NAMES), output_dict=True)

        #FOR BINARY
        
        # Binary results (No_Anemia vs Anemia) saved in BINARY_DIR
        binary_class_names = ["No_Anemia", "Anemia"]

        # Calibrated probs are computed later after temperature scaling,
        # but we can store y_true_bin now (already prepared globally as yte_bin).
        y_true_bin = yte_bin.astype(int).copy()


        cal_timing = {}
        scalerT = fit_temperature(model, cal_loader, timing_store=cal_timing)
        timing.update(cal_timing)

        te_probs_uncal = softmax_np(te["logits"])
        te_probs_cal = softmax_np(te["logits"] / scalerT.temperature())
        ece_uncal = expected_calibration_error(te_probs_uncal, te["y_true"])
        ece_cal = expected_calibration_error(te_probs_cal, te["y_true"])
        brier_cal = multiclass_brier(te_probs_cal, te["y_true"], cfg_local.NUM_CLASSES)

        #For Binary
        # Binary metrics + confusion matrices (raw + normalized) using calibrated probs
        # Binary probability of "Anemia" = 1 - P(class0)
        p_anemia_cal = 1.0 - te_probs_cal[:, 0]

        # Default threshold 0.50 (no other changes requested)
        y_pred_bin = (p_anemia_cal >= 0.50).astype(int)

        cm_bin = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
        cm_bin_norm = cm_bin / (cm_bin.sum(axis=1, keepdims=True) + 1e-12)

        # Save binary tables
        pd.DataFrame(cm_bin, index=binary_class_names, columns=binary_class_names).to_csv(
            BINARY_DIR / f"{tag}_binary_confusion_matrix.csv"
        )
        pd.DataFrame(cm_bin_norm, index=binary_class_names, columns=binary_class_names).to_csv(
            BINARY_DIR / f"{tag}_binary_confusion_matrix_normalized.csv"
        )

        rep_bin = classification_report(
            y_true_bin, y_pred_bin, target_names=binary_class_names, output_dict=True, zero_division=0
        )
        pd.DataFrame(rep_bin).T.to_csv(BINARY_DIR / f"{tag}_binary_classification_report.csv")

        # Extra binary scalar metrics
        bin_metrics = {
            "tag": tag,
            "threshold": 0.50,
            "binary_acc": float(accuracy_score(y_true_bin, y_pred_bin)),
            "binary_bal_acc": float(balanced_accuracy_score(y_true_bin, y_pred_bin)),
            "binary_f1": float(f1_score(y_true_bin, y_pred_bin, zero_division=0)),
            "binary_precision": float(precision_score(y_true_bin, y_pred_bin, zero_division=0)),
            "binary_recall_sensitivity": float(recall_score(y_true_bin, y_pred_bin, zero_division=0)),
            "binary_mcc": float(matthews_corrcoef(y_true_bin, y_pred_bin)),
            "binary_kappa": float(cohen_kappa_score(y_true_bin, y_pred_bin)),
        }
        pd.DataFrame([bin_metrics]).to_csv(BINARY_DIR / f"{tag}_binary_metrics.csv", index=False)

        # Save probability outputs (useful for ROC/PR externally)
        pd.DataFrame({
            "y_true_bin": y_true_bin,
            "p_anemia_cal": p_anemia_cal,
            "y_pred_bin_thr0.50": y_pred_bin
        }).to_csv(BINARY_DIR / f"{tag}_binary_probs_calibrated.csv", index=False)

        # Plot and save BOTH confusion matrices (raw + normalized) into binary directory
        fig = plot_confusion(cm_bin, binary_class_names, normalize=False, title=f"{tag}: Binary Confusion Matrix")
        save_png(fig, BINARY_DIR, f"{tag}_binary_cm"); plt.close(fig)

        fig = plot_confusion(cm_bin, binary_class_names, normalize=True, title=f"{tag}: Binary Confusion Matrix (Normalized)")
        save_png(fig, BINARY_DIR, f"{tag}_binary_cm_norm"); plt.close(fig)


        # Binary calibration plots (ECE + reliability) saved in BINARY_DIR
       
        # Binary prob of "Anemia"
        p_anemia_uncal = 1.0 - te_probs_uncal[:, 0]
        p_anemia_cal   = 1.0 - te_probs_cal[:, 0]

        # Convert to 2-class probability matrices for ECE/reliability helpers
        probs_bin_uncal = np.stack([1.0 - p_anemia_uncal, p_anemia_uncal], axis=1)
        probs_bin_cal   = np.stack([1.0 - p_anemia_cal,   p_anemia_cal],   axis=1)

        ece_bin_uncal = expected_calibration_error(probs_bin_uncal, y_true_bin, n_bins=15)
        ece_bin_cal   = expected_calibration_error(probs_bin_cal,   y_true_bin, n_bins=15)

        # Save binary ECE numbers
        pd.DataFrame([{
            "tag": tag,
            "ece_bin_uncal": float(ece_bin_uncal),
            "ece_bin_cal": float(ece_bin_cal),
            "temperature": float(scalerT.temperature()),
        }]).to_csv(BINARY_DIR / f"{tag}_binary_ece.csv", index=False)

        # Reliability plots (binary)
        bc_u, ba_u = reliability_bins(probs_bin_uncal, y_true_bin, n_bins=15)
        fig = plot_reliability(bc_u, ba_u, f"{tag}: Binary Reliability (Uncal)  ECE={ece_bin_uncal:.3f}")
        save_png(fig, BINARY_DIR, f"{tag}_binary_reliability_uncal"); plt.close(fig)

        bc_c, ba_c = reliability_bins(probs_bin_cal, y_true_bin, n_bins=15)
        fig = plot_reliability(bc_c, ba_c, f"{tag}: Binary Reliability (Cal)  T={scalerT.temperature():.3f}  ECE={ece_bin_cal:.3f}")
        save_png(fig, BINARY_DIR, f"{tag}_binary_reliability_cal"); plt.close(fig)




        
        with WallClock("decision_curve_sec", DEVICE, timing):
            p_anemia = 1.0 - te_probs_cal[:, 0]
            thresholds = np.linspace(0.01, 0.99, 99)
            dca_df = decision_curve_binary(y_true_bin=yte_bin, p_pos=p_anemia, thresholds=thresholds)
            dca_df.to_csv(TABLES_DIR / f"{tag}_decision_curve.csv", index=False)

        # XAI 
        with WallClock("xai_total_sec", DEVICE, timing):
            glob_attn = compute_global_attention_importance(model, test_loader, n_tokens=N_TOKENS, max_batches=30)
            per_cls_attn = per_class_attention_importance(model, test_loader, n_tokens=N_TOKENS, num_classes=cfg_local.NUM_CLASSES, max_batches=60)

            model.eval()
            attn_collect = []
            n_batches = 0
            for Xb, _ in test_loader:
                Xb = Xb.to(DEVICE)
                out_att = model(Xb, return_attention=True)
                attn_collect.append(out_att["attn_layers"])
                n_batches += 1
                if n_batches >= int(max(1, cfg_local.ATTN_VIZ_MAX_BATCHES)):
                    break

            attn_all_layers = []
            for layer_i in range(len(attn_collect[0])):
                mats = [attn_collect[b][layer_i] for b in range(len(attn_collect))]
                attn_all_layers.append(torch.cat(mats, dim=0))

            rollout = attention_rollout(attn_all_layers)
            rollout_receiver = rollout.sum(axis=0)

            baseline = Xtr.mean(axis=0)
            occ_imp = occlusion_sensitivity(
                model, Xte, yte, baseline=baseline,
                n_tokens=N_TOKENS, n_samples=cfg_local.OCC_SAMPLES
            )

            ig_base = np.zeros_like(baseline) if cfg_local.IG_BASELINE.lower().strip() == "zero" else baseline.copy()
            ig_all, ig_sample_attr, ig_sample_meta = compute_ig_pack(
                model=model,
                X_np=Xte,
                y_true=yte,
                baseline=ig_base,
                n_samples=cfg_local.IG_SAMPLES,
                steps=cfg_local.IG_STEPS
            )
            ig_sample_id = int(ig_sample_meta[0])
            ig_sample_true = int(ig_sample_meta[1])

            with torch.no_grad():
                ig_pred = int(
                    model(torch.tensor(Xte[ig_sample_id], dtype=torch.float32, device=DEVICE).unsqueeze(0))["subtype"]
                    .argmax(dim=1).item()
                )

            # entropy saved per layer
            layer_artifacts = []
            n_layers = len(attn_all_layers)
            for layer_idx in range(n_layers):
                attn_layer_bhff = attn_all_layers[layer_idx]
                attn_layer_hff = attn_layer_bhff.mean(dim=0)           # [H,F,F]
                attn_layer_hff_np = attn_layer_hff.detach().cpu().numpy()
                attn_headavg_ff = attn_layer_hff_np.mean(axis=0)        # [F,F]
                ent = attention_entropy_from_layer(attn_bhff=attn_layer_bhff)  # [H,F]

                layer_artifacts.append({
                    "layer_idx": int(layer_idx),
                    "attn_hff": attn_layer_hff_np,
                    "attn_avg_ff": attn_headavg_ff,
                    "entropy_hf": ent
                })

                ent_df = pd.DataFrame(ent, columns=FEATURES)
                ent_df.insert(0, "head", [f"H{i+1}" for i in range(ent.shape[0])])
                ent_df.to_csv(TABLES_DIR / f"{tag}_attention_entropy_layer{layer_idx+1}.csv", index=False)

                if getattr(cfg_local, "SAVE_RAW_ATTN_MATRICES", False):
                    np.savez_compressed(
                        TABLES_DIR / f"{tag}_L{layer_idx+1}_attn_matrices.npz",
                        attn_hff=attn_layer_hff_np,
                        attn_avg_ff=attn_headavg_ff,
                        entropy_hf=ent
                    )

        # Inference timing
        with WallClock("inference_repeated_total_sec", DEVICE, timing):
            model.eval()
            reps = int(max(1, cfg_local.TIME_INFERENCE_REPEATS))
            n_seen = 0
            for _ in range(reps):
                for X, _ in test_loader:
                    X = X.to(DEVICE)
                    _ = model(X, return_attention=False)
                    n_seen += int(X.size(0))
            timing["inference_repeated_samples"] = float(n_seen)

        # Save checkpoint
        ckpt = MODELS_DIR / f"{tag}_best.pt"
        torch.save({
            "state_dict": model.state_dict(),
            "loss_state_dict": loss_fn.state_dict(),
            "features": FEATURES,
            "scaler_mean": scaler.mean_,
            "scaler_scale": scaler.scale_,
            "cfg": asdict(cfg_local),
            "temperature": scalerT.temperature(),
            "tag": tag,
            "overrides": overrides,
        }, ckpt)

        # Save tables/metrics
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(METRICS_DIR / f"{tag}_history.csv", index=False)

        pd.DataFrame(cm, index=cfg_local.CLASS_NAMES, columns=cfg_local.CLASS_NAMES).to_csv(TABLES_DIR / f"{tag}_confusion_matrix.csv")
        cm_norm = cm / (cm.sum(axis=1, keepdims=True) + 1e-12)
        pd.DataFrame(cm_norm, index=cfg_local.CLASS_NAMES, columns=cfg_local.CLASS_NAMES).to_csv(
            TABLES_DIR / f"{tag}_confusion_matrix_normalized.csv"
        )
        pd.DataFrame(rep).T.to_csv(TABLES_DIR / f"{tag}_per_class_report.csv")

        per_class_df = per_class_from_cm(cm, list(cfg_local.CLASS_NAMES))
        per_class_df.to_csv(TABLES_DIR / f"{tag}_per_class_metrics.csv", index=False)

        pd.DataFrame({"feature": FEATURES, "attention_global": glob_attn}).to_csv(TABLES_DIR / f"{tag}_attention_global.csv", index=False)
        pc = pd.DataFrame({cfg_local.CLASS_NAMES[c]: per_cls_attn[c] for c in range(cfg_local.NUM_CLASSES)}, index=FEATURES)
        pc.index.name = "feature"
        pc.to_csv(TABLES_DIR / f"{tag}_attention_per_class.csv")

        pd.DataFrame({"feature": FEATURES, "occlusion_importance": occ_imp}).to_csv(TABLES_DIR / f"{tag}_occlusion.csv", index=False)
        pd.DataFrame({"feature": FEATURES, "rollout_receiver_score": rollout_receiver}).to_csv(TABLES_DIR / f"{tag}_rollout_receiver_score.csv", index=False)

        pd.DataFrame(ig_all, columns=FEATURES).to_csv(TABLES_DIR / f"{tag}_integrated_gradients_samples.csv", index=False)
        pd.DataFrame({"feature": FEATURES, "ig_sample_attr": ig_sample_attr}).to_csv(TABLES_DIR / f"{tag}_integrated_gradients_sample_{ig_sample_id}.csv", index=False)

        auto_final = loss_fn.autoW.snapshot() if hasattr(loss_fn, "autoW") else {}

        test_metrics = {
            "tag": tag,
            "test_acc": te["acc"],
            "test_bal_acc": te["bal_acc"],
            "test_macro_f1": te["macro_f1"],
            "test_weighted_f1": te["weighted_f1"],
            "test_micro_f1": te["micro_f1"],
            "test_macro_precision": te["macro_precision"],
            "test_macro_recall_sensitivity": te["macro_recall"],
            "test_weighted_precision": te["weighted_precision"],
            "test_weighted_recall": te["weighted_recall"],
            "test_mcc": te["mcc"],
            "test_kappa": te["kappa"],
            "temperature": float(scalerT.temperature()),
            "ece_uncal": float(ece_uncal),
            "ece_cal": float(ece_cal),
            "brier_cal": float(brier_cal),
            "ckpt_path": str(ckpt),
            "stage_boundaries_epoch_global": stage_boundaries,
            "use_smote_raw_only": bool(cfg_local.USE_SMOTE),
            "xai_attn_layer_viz": int(cfg_local.ATTN_LAYER_VIZ),
            "ig_sample_id": int(ig_sample_id),
            "ig_sample_true": int(ig_sample_true),
            "ig_sample_pred": int(ig_pred),
            "stage1_classes": list(cfg_local.STAGE1_CLASSES),
            "stage2_classes": list(cfg_local.STAGE2_CLASSES),
            "stage3_classes": list(cfg_local.STAGE3_CLASSES),
        }
        pd.DataFrame([test_metrics]).to_csv(TABLES_DIR / f"{tag}_results_table.csv", index=False)

        # Save timing csv now; JSON built AFTER WallClock exits 
        pd.DataFrame(timing_rows).to_csv(METRICS_DIR / f"{tag}_timing.csv", index=False)
        if len(batch_profile_rows) > 0:
            pd.DataFrame(batch_profile_rows).to_csv(METRICS_DIR / f"{tag}_batch_profile.csv", index=False)

        # PDF report
        run_pdf = PLOTS_DIR / f"{tag}_PLOTS.pdf"
        with PdfPages(run_pdf) as pdf:
            if len(hist_df) > 0:
                fig = plot_lines(hist_df, "epoch_global", ["tr_total", "va_total"],
                                 ["Train total loss", "Val total loss"],
                                 f"{tag}: Total Loss (Train vs Val)", "Global epoch", "Loss",
                                 stage_boundaries=stage_boundaries if use_curriculum else None)
                save_png(fig, PLOTS_DIR, f"{tag}_loss_total"); pdf.savefig(fig); plt.close(fig)

                fig = plot_lines(hist_df, "epoch_global", ["tr_sub", "va_sub"],
                                 ["Train subtype loss", "Val subtype loss"],
                                 f"{tag}: Subtype Loss (Train vs Val)", "Global epoch", "Loss",
                                 stage_boundaries=stage_boundaries if use_curriculum else None)
                save_png(fig, PLOTS_DIR, f"{tag}_loss_subtype"); pdf.savefig(fig); plt.close(fig)

                fig, ax = _make_fig()
                ax.plot(hist_df["epoch_global"], hist_df["tr_acc_pct"],
                        linewidth=PLOT.line_width, label="Train accuracy (%)")
                ax.plot(hist_df["epoch_global"], 100.0 * hist_df["va_acc"],
                        linewidth=PLOT.line_width, label="Val accuracy (%)")
                for b in (stage_boundaries if (use_curriculum and stage_boundaries) else []):
                    ax.axvline(b, linestyle="--", linewidth=PLOT.vline_width, alpha=0.65)
                _bold_title_labels(ax, f"{tag}: Accuracy (Train vs Val)", "Global epoch", "Accuracy (%)")
                _bold_legend(ax)
                style_axes(ax)
                fig.tight_layout()
                save_png(fig, PLOTS_DIR, f"{tag}_train_vs_val_acc_pct"); pdf.savefig(fig); plt.close(fig)

                fig, ax = _make_fig()
                ax.plot(hist_df["epoch_global"], hist_df["tr_macro_f1"],
                        linewidth=PLOT.line_width, label="Train macro-F1")
                ax.plot(hist_df["epoch_global"], hist_df["va_macro_f1"],
                        linewidth=PLOT.line_width, label="Val macro-F1")
                for b in (stage_boundaries if (use_curriculum and stage_boundaries) else []):
                    ax.axvline(b, linestyle="--", linewidth=PLOT.vline_width, alpha=0.65)
                _bold_title_labels(ax, f"{tag}: Macro-F1 (Train vs Val)", "Global epoch", "Macro-F1")
                _bold_legend(ax)
                style_axes(ax)
                fig.tight_layout()
                save_png(fig, PLOTS_DIR, f"{tag}_train_vs_val_macroF1"); pdf.savefig(fig); plt.close(fig)

                fig = plot_lines(hist_df, "epoch_global", ["lr"], ["LR"],
                                 f"{tag}: Learning Rate Schedule (global stage-aware)",
                                 "Global epoch", "LR",
                                 stage_boundaries=stage_boundaries if use_curriculum else None)
                save_png(fig, PLOTS_DIR, f"{tag}_lr"); pdf.savefig(fig); plt.close(fig)

                fig = plot_auto_weights(hist_df, tag=tag, stage_boundaries=stage_boundaries if use_curriculum else None)
                if fig is not None:
                    save_png(fig, PLOTS_DIR, f"{tag}_auto_task_weights"); pdf.savefig(fig); plt.close(fig)

            fig = plot_confusion(cm, list(cfg_local.CLASS_NAMES), normalize=False, title=f"{tag}: Confusion Matrix")
            save_png(fig, PLOTS_DIR, f"{tag}_cm"); pdf.savefig(fig); plt.close(fig)

            fig = plot_confusion(cm, list(cfg_local.CLASS_NAMES), normalize=True, title=f"{tag}: Confusion Matrix (Normalized)")
            save_png(fig, PLOTS_DIR, f"{tag}_cm_norm"); pdf.savefig(fig); plt.close(fig)

             # Binary confusion matrices also in the PDF
            fig = plot_confusion(cm_bin, binary_class_names, normalize=False, title=f"{tag}: Binary Confusion Matrix")
            pdf.savefig(fig); plt.close(fig)

            fig = plot_confusion(cm_bin, binary_class_names, normalize=True, title=f"{tag}: Binary Confusion Matrix (Normalized)")
            pdf.savefig(fig); plt.close(fig)

            fig = plot_per_class_f1(rep, list(cfg_local.CLASS_NAMES), f"{tag}: Per-class F1")
            save_png(fig, PLOTS_DIR, f"{tag}_perclass_f1"); pdf.savefig(fig); plt.close(fig)

            bc1, ba1 = reliability_bins(te_probs_uncal, te["y_true"], n_bins=15)
            fig = plot_reliability(bc1, ba1, f"{tag}: Reliability (Uncal)  ECE={ece_uncal:.3f}")
            save_png(fig, PLOTS_DIR, f"{tag}_reliability_uncal"); pdf.savefig(fig); plt.close(fig)

            bc2, ba2 = reliability_bins(te_probs_cal, te["y_true"], n_bins=15)
            fig = plot_reliability(bc2, ba2, f"{tag}: Reliability (Cal)  T={scalerT.temperature():.3f}  ECE={ece_cal:.3f}")
            save_png(fig, PLOTS_DIR, f"{tag}_reliability_cal"); pdf.savefig(fig); plt.close(fig)

            fig, ax = _make_fig(size=(7.4, 6.2))
            for c in range(cfg_local.NUM_CLASSES):
                y_bin_c = (te["y_true"] == c).astype(int)
                fpr, tpr, _ = roc_curve(y_bin_c, te_probs_cal[:, c])
                ax.plot(fpr, tpr, linewidth=PLOT.line_width, label=f"{cfg_local.CLASS_NAMES[c]} AUC={auc(fpr, tpr):.3f}")
            ax.plot([0, 1], [0, 1], '--', linewidth=PLOT.vline_width)
            _bold_title_labels(ax, f"{tag}: ROC (OvR, Calibrated)", "False Positive Rate", "True Positive Rate")
            _bold_legend(ax)
            style_axes(ax)
            fig.tight_layout()
            save_png(fig, PLOTS_DIR, f"{tag}_roc_ovr"); pdf.savefig(fig); plt.close(fig)

            fig, ax = _make_fig(size=(7.4, 6.2))
            for c in range(cfg_local.NUM_CLASSES):
                y_bin_c = (te["y_true"] == c).astype(int)
                prec, rec, _ = precision_recall_curve(y_bin_c, te_probs_cal[:, c])
                ap = average_precision_score(y_bin_c, te_probs_cal[:, c])
                ax.plot(rec, prec, linewidth=PLOT.line_width, label=f"{cfg_local.CLASS_NAMES[c]} AP={ap:.3f}")
            _bold_title_labels(ax, f"{tag}: Precision–Recall (OvR, Calibrated)", "Recall", "Precision")
            _bold_legend(ax)
            style_axes(ax)
            fig.tight_layout()
            save_png(fig, PLOTS_DIR, f"{tag}_pr_ovr"); pdf.savefig(fig); plt.close(fig)

            fig = plot_decision_curve(dca_df, f"{tag}: Decision Curve Analysis (Binary Anemia)")
            save_png(fig, PLOTS_DIR, f"{tag}_decision_curve"); pdf.savefig(fig); plt.close(fig)

            fig = plot_barh_top(glob_attn, FEATURES, f"{tag}: Global Pooling Attention (Top-15)", topk=15)
            save_png(fig, PLOTS_DIR, f"{tag}_xai_global_attention_top15"); pdf.savefig(fig); plt.close(fig)

            fig = plot_barh_top(occ_imp, FEATURES, f"{tag}: Occlusion Sensitivity (Top-15)", topk=15)
            save_png(fig, PLOTS_DIR, f"{tag}_xai_occlusion_top15"); pdf.savefig(fig); plt.close(fig)

            fig = plot_barh_top(rollout_receiver, FEATURES, f"{tag}: Attention Rollout Receiver (Top-15)", topk=15)
            save_png(fig, PLOTS_DIR, f"{tag}_xai_rollout_receiver_top15"); pdf.savefig(fig); plt.close(fig)

            fig = plot_integrated_gradients_sample(
                ig_sample_attr, FEATURES,
                sample_id=ig_sample_id,
                true_class=ig_sample_true,
                pred_class=ig_pred,
                class_names=list(cfg_local.CLASS_NAMES)
            )
            save_png(fig, PLOTS_DIR, f"{tag}_ig_sample_{ig_sample_id}"); pdf.savefig(fig); plt.close(fig)

            fig = plot_integrated_gradients_global(
                ig_all, FEATURES,
                title=f"{tag}: Integrated Gradients (Global Mean |Attr|, N={ig_all.shape[0]})"
            )
            save_png(fig, PLOTS_DIR, f"{tag}_ig_global"); pdf.savefig(fig); plt.close(fig)

            fig = plot_attention_rollout_matrix(rollout, FEATURES, title=f"{tag}: Attention Rollout Matrix")
            save_png(fig, PLOTS_DIR, f"{tag}_attn_rollout_matrix"); pdf.savefig(fig); plt.close(fig)

            if getattr(cfg_local, "SAVE_ALL_LAYERS_ATTENTION", True):
                for item in layer_artifacts:
                    L = int(item["layer_idx"]) + 1
                    attn_hff = item["attn_hff"]
                    ent_hf = item["entropy_hf"]

                    fig = plot_multihead_attention_grid(attn_hff, FEATURES, layer_idx=L-1, n_heads=cfg_local.N_HEADS)
                    save_png(fig, PLOTS_DIR, f"{tag}_L{L}_attn_grid"); pdf.savefig(fig); plt.close(fig)

                    fig = plot_attention_heatmap_averaged(attn_hff, FEATURES, title=f"{tag}: Layer {L} Attention (Head-Averaged)")
                    save_png(fig, PLOTS_DIR, f"{tag}_L{L}_attn_heatmap_avg"); pdf.savefig(fig); plt.close(fig)

                    fig = plot_attention_entropy_heatmap(ent_hf, FEATURES, layer_idx=L-1)
                    save_png(fig, PLOTS_DIR, f"{tag}_L{L}_attn_entropy"); pdf.savefig(fig); plt.close(fig)

        summary = dict(test_metrics)
        summary["cfg_overrides"] = overrides
        summary["auto_weights_final"] = auto_final
        summary["run_pdf"] = str(run_pdf)

   
    # Build timing_summary AFTER exiting total_run_wallclock_sec context
    timing_summary = {
        "device": str(DEVICE),
        "amp": bool(DEVICE.type == "cuda"),

        "n_train_raw_before_smote": int(len(train_df_raw)),
        "n_train_after_smote": int(len(train_df)),
        "n_val_eval": int(len(val_eval_df)),
        "n_cal": int(len(cal_df)),
        "n_test": int(len(test_df)),

        "use_smote_raw_only": bool(cfg_local.USE_SMOTE),

        "n_tokens": int(N_TOKENS),
        "n_layers": int(cfg_local.N_LAYERS),
        "d_model": int(cfg_local.D_MODEL),
        "d_ff": int(cfg_local.D_FF),
        "n_heads": int(cfg_local.N_HEADS),

        "n_params_trainable_model_plus_autoweights": int(n_params),
        "approx_param_memory_bytes_fp32": int(param_bytes),

        "time_total_run_sec": float(timing.get("total_run_wallclock_sec", np.nan)),
        "time_stage1_sec": float(timing.get("stage_1_total_sec", 0.0)),
        "time_stage2_sec": float(timing.get("stage_2_total_sec", 0.0)),
        "time_stage3_sec": float(timing.get("stage_3_total_sec", 0.0)),

        "time_test_eval_sec": float(timing.get("test_eval_sec", np.nan)),
        "time_temperature_fit_sec": float(timing.get("calibration_temperature_fit_sec", np.nan)),
        "time_decision_curve_sec": float(timing.get("decision_curve_sec", np.nan)),
        "time_xai_total_sec": float(timing.get("xai_total_sec", np.nan)),

        "time_inference_repeated_total_sec": float(timing.get("inference_repeated_total_sec", np.nan)),
        "inference_repeated_samples": float(timing.get("inference_repeated_samples", 0.0)),
        "complexity_bigO_per_layer": complexity["bigO_per_layer"],
        "complexity_bigO_total": complexity["bigO_total"],
        "approx_flops_total_per_sample": float(complexity["approx_flops_total_per_sample"]),

        "stage1_classes": list(cfg_local.STAGE1_CLASSES),
        "stage2_classes": list(cfg_local.STAGE2_CLASSES),
        "stage3_classes": list(cfg_local.STAGE3_CLASSES),
    }

    if timing_summary["inference_repeated_samples"] > 0 and timing_summary["time_inference_repeated_total_sec"] > 0:
        timing_summary["inference_time_ms_per_sample"] = float(
            1000.0 * timing_summary["time_inference_repeated_total_sec"] / timing_summary["inference_repeated_samples"]
        )
    else:
        timing_summary["inference_time_ms_per_sample"] = float("nan")

    with open(METRICS_DIR / f"{tag}_timing.json", "w") as f:
        json.dump(timing_summary, f, indent=2)
    pd.DataFrame([timing_summary]).to_csv(TABLES_DIR / f"{tag}_timing_table.csv", index=False)

    final_summary = dict(summary)
    final_summary["timing_summary"] = timing_summary
    with open(METRICS_DIR / f"{tag}_summary.json", "w") as f:
        json.dump(final_summary, f, indent=2)

    return {"summary": final_summary, "run_pdf": str(run_pdf)}


# RUN
MAIN_TAG = "MTFTT"

out = run_proposed_model(
    tag=MAIN_TAG,
    overrides=dict(),
    use_curriculum=cfg.USE_CURRICULUM,
    use_aug=cfg.USE_AUG,
    use_focaldice=cfg.USE_FOCALDICE,
    use_multitask=cfg.USE_MULTITASK,
    use_attn_reg=cfg.USE_ATTN_REG,
    epoch_scale=1.0
)

print("\n=== PROPOSED MODEL SUMMARY ===")
print(json.dumps(out["summary"], indent=2))

print("\n==================== DONE ====================")
print("Outputs saved in:", RUN_DIR)
print(" - Plots  :", PLOTS_DIR)
print(" - Metrics:", METRICS_DIR)
print(" - Tables :", TABLES_DIR)
print(" - Models :", MODELS_DIR)
print("Run PDF:", out["run_pdf"])
print("Complexity file:", METRICS_DIR / f"{MAIN_TAG}_complexity.json")


DEVICE: cuda
RUN_DIR: Final_MTFTT_OUTPUTS\V2.2Bin_MTFTT_run_2026-02-10_at_15-34-32

=== STAGE CLASS CONTROL (TRAIN + VAL) ===
Stage-1 classes: (0, 1, 2) => ['No_Anemia', 'HGB_Anemia', 'Iron_Def_Anemia']
Stage-2 classes: (0, 1, 2, 4) => ['No_Anemia', 'HGB_Anemia', 'Iron_Def_Anemia', 'B12_Def_Anemia']
Stage-3 classes: (0, 1, 2, 3, 4) => ['No_Anemia', 'HGB_Anemia', 'Iron_Def_Anemia', 'Folate_Def_Anemia', 'B12_Def_Anemia']

Label ↔ flags consistency OK
CSV shape: (15300, 29)
Class counts:
 All_Class
0    9747
1    1019
2    4182
3     153
4     199
Name: count, dtype: int64

Split sizes (RAW):
Train: 10709 Val_eval: 1147 Cal: 1148 Test: 2296

[SMOTE=ON] Train size before: 10709 after: 34110
[SMOTE=ON] Train class counts after:
 All_Class
0    6822
1    6822
2    6822
3    6822
4    6822
Name: count, dtype: int64

Total tokens: 32
Train/Val/Cal/Test shapes after FE: (34110, 33) (1147, 37) (1148, 37) (2296, 37)
[VAL ] Stage1 using classes=(0, 1, 2) (['No_Anemia', 'HGB_Anemia', 'Iron_Def_Anem

  0%|                                                                                           | 0/80 [00:00<?, ?it/s]C:\Users\dpnsh\AppData\Local\Temp\ipykernel_15224\2026307764.py:1636: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


[MTFTT] Stage1 Ep5 | val_macroF1=0.7183 bal_acc=0.7898 val_loss=35.3591 lr=5.75e-05


[MTFTT] Stage1 Ep10 | val_macroF1=0.9218 bal_acc=0.9647 val_loss=17.5396 lr=1.15e-04


[MTFTT] Stage1 Ep15 | val_macroF1=0.9354 bal_acc=0.9592 val_loss=5.8736 lr=1.72e-04


...

[MTFTT] Stage3 Ep10 | val_macroF1=0.9532 bal_acc=0.9684 val_loss=-2.0629 lr=6.33e-05


[MTFTT] Stage3 Ep15 | val_macroF1=0.9669 bal_acc=0.9818 val_loss=-1.5550 lr=5.69e-05


[MTFTT] Stage3 Ep20 | val_macroF1=0.9628 bal_acc=0.9802 val_loss=-1.6038 lr=5.04e-05


[MTFTT] Stage3 Ep25 | val_macroF1=0.9628 bal_acc=0.9802 val_loss=-1.6750 lr=4.38e-05


[MTFTT] Early stopping triggered at Stage3 Ep27 (patience=12)

=== PROPOSED MODEL SUMMARY ===
{
  "tag": "MTFTT",
  "test_acc": 0.9817073170731707,
  "test_bal_acc": 0.9260202214682474,
  "test_macro_f1": 0.9120470295083812,
  "test_weighted_f1": 0.9819518069790208,
  "test_micro_f1": 0.9817073170731707,
  "test_macro_precision": 0.899163598963604,
  "test_macro_recall_sensitivity": 0.9260202214682474,
  "test_weighted_precision": 0.9824204994389035,
  "test_weighted_recall": 0.9817073170731707,
  "test_mcc": 0.9645740841282694,
  "test_kappa": 0.9645184648805214,
  "temperature": 0.6096363663673401,
  "ece_uncal": 0.026160128388670677,
  "ece_cal": 0.004212014623739163,
  "brier_cal": 0.02664736463142476,
  "ckpt_path": "Final_MTFTT_OUTPUTS\\V2.2Bin_MTFTT_run_2026-02-10_at_15-34-32\\models\\MTFTT_best.pt",
  "stage_boundaries_epoch_global": [
    30,
    70,
    97
  ],
  "use_smote_raw_only": true,
  "xai_attn_layer_viz": -1,
  "ig_sample_id": 886,
  "ig_sample_true": 0,
  "ig_sample